<a href="https://colab.research.google.com/github/gurveervirk/CodeSage/blob/main/using_inspect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Dependencies

In [ ]:
!pip install -q importlib beautifulsoup4 typing llama-index-graph-stores-neo4j importlib-metadata astroid llama-index-llms-gemini llama-index-embeddings-gemini llama-index-embeddings-huggingface ratelimit llama-index-embeddings-openai llama-index-vector-stores-neo4jvector llama-index-readers-web llama-index-tools-duckduckgo llama-index-agent-introspective

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 82.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.2/275.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.7/301.7 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 19.5 MB/s eta 0:

In [ ]:
!playwright install --with-deps chromium

Installing dependencies...
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [45.2 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,227 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [3,527 kB]
Get:

# Install a module for demo

In [ ]:
!pip install -q bpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.1/390.1 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 55.3 MB/s eta 0:00:00


In [ ]:
import inspect
import bpy

doc = inspect.getdoc(bpy)

In [ ]:
doc

'Give access to blender data and utility functions.'

# Test Neo4jPropertyGraphStore with support for Full text search


In [ ]:
def remove_empty_values(input_dict):
    """
    Remove entries with empty values from the dictionary.

    Parameters:
    input_dict (dict): The dictionary from which empty values need to be removed.

    Returns:
    dict: A new dictionary with all empty values removed.
    """
    # Create a new dictionary excluding empty values
    return {key: value for key, value in input_dict.items() if value}

In [ ]:
from typing import Any, Dict, List, Optional, Tuple, Union
import logging
import llama_index.vector_stores.neo4jvector.base as nvs
import neo4j
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.core.bridge.pydantic import PrivateAttr
from llama_index.core.vector_stores.types import (
    VectorStoreQuery
)
from llama_index.core.graph_stores.types import (
    LabelledNode
)
from llama_index.core.vector_stores.utils import (
    metadata_dict_to_node,
)

_logger = logging.getLogger(__name__)
BASE_ENTITY_LABEL = "__Entity__"
BASE_NODE_LABEL = "__Node__"

def check_if_not_null(props: List[str], values: List[Any]) -> None:
    """Check if variable is not null and raise error accordingly."""
    for prop, value in zip(props, values):
        if not value:
            raise ValueError(f"Parameter `{prop}` must not be None or empty string")

def parse_operator(operator: FilterOperator) -> str:
    if operator == FilterOperator.EQ:
        return "="
    elif operator == FilterOperator.IN:
        return "IN"
    else:
        _logger.warning(f"Unknown operator: {operator}, fallback to '='")
        return "="

class Neo4jPropertyGraphStoreWithFTS(Neo4jPropertyGraphStore):
    stores_text: bool = True
    flat_metadata: bool = True
    distance_strategy: str
    index_name: str
    keyword_index_name: str
    hybrid_search: bool
    node_label: str
    embedding_node_property: str
    text_node_property: str
    retrieval_query: str
    _support_metadata_filter: bool = PrivateAttr()
    _is_enterprise: bool = PrivateAttr()
    custom_fts_indices: Any = []

    def __init__(
        self,
        username: str,
        password: str,
        url: str,
        database: Optional[str] = "neo4j",
        index_name: str = "entity",
        keyword_index_name: str = "keyword",
        node_label: str = "__Node__",
        embedding_node_property: str = "embedding",
        text_node_property: str = "summary",
        add_custom_fts_indices: Union[List[str], Dict[str, str], Dict[str, Tuple[str, List[str]]]] = None
        hybrid_search: bool = True,
        retrieval_query: str = "",
        refresh_schema: bool = True,
        sanitize_query_output: bool = True,
        enhanced_schema: bool = False,
        **neo4j_kwargs: Any,
    ) -> None:
        self.index_name = index_name
        self.keyword_index_name = keyword_index_name
        self.node_label = node_label
        self.embedding_node_property = embedding_node_property
        self.text_node_property = text_node_property
        self.hybrid_search = hybrid_search
        self.retrieval_query = retrieval_query
        self.database = database
        super().__init__(
            username,
            password,
            url,
            database,
            refresh_schema,
            sanitize_query_output,
            enhanced_schema,
            **neo4j_kwargs,
        )
        check_if_not_null(
            [
                "index_name",
                "node_label",
                "embedding_node_property",
                "text_node_property",
            ],
            [index_name, node_label, embedding_node_property, text_node_property],
        )
        if add_custom_fts_indices and not hybrid_search:
            raise ValueError(
                "Custom FTS indices can only be used with hybrid search enabled"
            )

        if hybrid_search:
            fts_node_label = self.retrieve_existing_fts_index()
            # If the FTS index doesn't exist yet
            if not fts_node_label:
                self.create_new_keyword_index()
            else:  # Validate that FTS and Vector index use the same information
                if not fts_node_label == self.node_label:
                    raise ValueError(
                        "Vector and keyword index don't index the same node label"
                    )

            # Retrieve existing custom FTS indices
            self.custom_fts_indices = self.retrieve_existing_custom_fts_indices()

            # Add custom FTS indices if provided
            if add_custom_fts_indices:
              # Store existing FTS indices' names for matching
              custom_fts_indices_names = set()
              for custom_fts_index in self.custom_fts_indices:
                custom_fts_indices_names.add(custom_fts_index["name"])

              if isinstance(add_custom_fts_indices, list):
                # Use same string for custom index name and node label (as property graph store is catered towards ER structure)
                for index_name in add_custom_fts_indices:
                  if index_name in custom_fts_indices_names:
                    # Log name already exists
                    _logger.warning(f"{index_name} already exists")
                    continue
                  self.create_new_keyword_index(index_name=index_name, node_label=index_name)
              elif isinstance(add_custom_fts_indices, dict):
                for index_name, data in add_custom_fts_indices.items():
                  if index_name in custom_fts_indices_names:
                    # Log name already exists
                    _logger.warning(f"{index_name} already exists")
                    continue
                  if isinstance(data, str):
                    self.create_new_keyword_index(index_name=index_name, node_label=data)
                  else:
                    node_label, text_node_properties = data
                    self.create_new_keyword_index(index_name=index_name, node_label=node_label, text_node_properties=text_node_properties)
              else:
                raise ValueError(f"Invalid type for add_custom_fts_indices: {type(add_custom_fts_indices)}")

              self.custom_fts_indices = self.retrieve_existing_custom_fts_indices()

        # Improve structure of custom_fts_indices from list of dictionaries to a single dictionary with key being index_name
        structured_custom_fts_indices = {}

        for custom_fts_index in self.custom_fts_indices:
          structured_custom_fts_indices[custom_fts_index["name"]] = {
              "label": custom_fts_index["labelsOrTypes"][0],
              "properties": custom_fts_index["properties"]
          }

        self.custom_fts_indices_by_name = structured_custom_fts_indices

        # Improve structure of custom_fts_indices from list of dictionaries to a single dictionary with key being label
        structured_custom_fts_indices = {}

        for custom_fts_index in self.custom_fts_indices:
          if custom_fts_index["labelsOrTypes"][0] not in structured_custom_fts_indices:
            structured_custom_fts_indices[custom_fts_index["labelsOrTypes"][0]] = []
          structured_custom_fts_indices[custom_fts_index["labelsOrTypes"][0]].append({
              "name": custom_fts_index["name"],
              "properties": custom_fts_index["properties"]
          })

        self.custom_fts_indices_by_label = structured_custom_fts_indices

    def create_new_keyword_index(self, node_label: str = None, index_name: str = None, text_node_properties: List[str] = []) -> None:
        """
        This method constructs a Cypher query and executes it
        to create a new full text index in Neo4j.
        """
        node_props = text_node_properties or [self.text_node_property]
        name = index_name or self.keyword_index_name
        node_label = node_label or self.node_label
        fts_index_query = (
            f"CREATE FULLTEXT INDEX {name} "
            f"FOR (n:`{node_label}`) ON EACH "
            f"[{', '.join(['n.`' + el + '`' for el in node_props])}]"
        )
        self.structured_query(fts_index_query)

    def retrieve_existing_fts_index(self) -> Optional[str]:
        """Check if the fulltext index exists in the Neo4j database.

        This method queries the Neo4j database for existing fts indexes
        with the specified name.

        Returns:
            (Tuple): keyword index information
        """
        index_information = self.structured_query(
            "SHOW INDEXES YIELD name, type, labelsOrTypes, properties, options "
            "WHERE type = 'FULLTEXT' AND (name = $keyword_index_name "
            "OR (labelsOrTypes = [$node_label] AND "
            "properties = $text_node_property)) "
            "RETURN name, labelsOrTypes, properties, options ",
            param_map={
                "keyword_index_name": self.keyword_index_name,
                "node_label": self.node_label,
                "text_node_property": self.text_node_property,
            },
        )
        # sort by index_name
        index_information = nvs.sort_by_index_name(index_information, self.index_name)
        try:
            self.keyword_index_name = index_information[0]["name"]
            self.text_node_property = index_information[0]["properties"][0]
            return index_information[0]["labelsOrTypes"][0]
        except IndexError:
            return None

    def retrieve_existing_custom_fts_indices(self) -> Optional[str]:
        """Retrieve existing custom FTS indices in the Neo4j database.

        This method queries the Neo4j database for existing custom fts
        indexes.

        Returns:
            (Tuple): keyword index information
        """
        indices_information = self.structured_query(
            "SHOW INDEXES YIELD name, type, labelsOrTypes, properties "
            "WHERE type = 'FULLTEXT' "
            "RETURN name, labelsOrTypes, properties ",
        )
        return indices_information

    def vector_query(self, query: VectorStoreQuery, **kwargs: Any) -> Tuple[List[LabelledNode], List[float]]:
        if query.filters:
            # Verify that 5.18 or later is used
            if not self._support_metadata_filter:
                raise ValueError(
                    "Metadata filtering is only supported in "
                    "Neo4j version 5.18 or greater"
                )

            parallel_query = (
                "CYPHER runtime = parallel parallelRuntimeSupport=all "
                if self._is_enterprise
                else ""
            )
            base_index_query = parallel_query + (
                f"MATCH (n:`{self.node_label}`) WHERE "
                f"n.`{self.embedding_node_property}` IS NOT NULL AND "
            )
            base_cosine_query = (
                " WITH n as node, vector.similarity.cosine("
                f"n.`{self.embedding_node_property}`, "
                "$embedding) AS score ORDER BY score DESC LIMIT toInteger($k) "
            )
            base_cosine_and_retrieval_query = (
                " WITH n as node, vector.similarity.cosine("
                f"n.`{self.embedding_node_property}`, "
                "$embedding) AS score ORDER BY score DESC LIMIT toInteger($k) "
                "RETURN node, score"
            )

            search_queries = []
            filters = query.filters.filters

            if self.hybrid_search and self.custom_fts_indices:
              # Considering only the following 2 types of metadata filtering in Property Graph Stores as it should encompass most, if not all, use cases pretty well
              for filter in filters:
                key = filter.key.upper()
                values = filter.value
                if isinstance(values, str):
                  values = [values]
                for value in values:
                  if key == "NAME":
                    if value in self.custom_fts_indices_by_name:
                      custom_fts_query = (
                          f"CALL db.index.fulltext.queryNodes({value},"
                          " $query, {limit: $k}) "
                          "YIELD node, score "
                          "WITH collect({node:node, score:score}) AS nodes, max(score) AS max "
                          "UNWIND nodes AS n "
                          "RETURN n.node AS node, (n.score / max) AS score UNION "
                      )

                      label = self.custom_fts_indices_by_name[value]["label"]
                      custom_index_query = parallel_query + (
                          f"MATCH (n:`{label}`) WHERE "
                          f"n.`{self.embedding_node_property}` IS NOT NULL "
                      )

                      final_query = (
                        "CALL { "
                        f"{custom_fts_query} "
                        f"{custom_index_query} "
                        f"{base_cosine_and_retrieval_query} "
                        "} "
                        "RETURN node, score "
                      )

                      search_queries.append(final_query)

                  elif key == "LABEL":
                    custom_fts_query = ""

                    for index in self.custom_fts_indices_by_label[value]:
                      custom_fts_query += (
                          f"CALL db.index.fulltext.queryNodes({index["name"]},"
                          " $query, {limit: $k}) "
                          "YIELD node, score "
                          "WITH collect({node:node, score:score}) AS nodes, max(score) AS max "
                          "UNWIND nodes AS n "
                          "RETURN n.node AS node, (n.score / max) AS score UNION \n"
                      )

                    label = value
                    custom_index_query = parallel_query + (
                      f"MATCH (n:`{label}`) WHERE "
                      f"n.`{self.embedding_node_property}` IS NOT NULL "
                    )

                    final_query = (
                      "CALL { "
                      f"{custom_fts_query} "
                      f"{custom_index_query} "
                      f"{base_cosine_and_retrieval_query} "
                      "} "
                      "RETURN node, score "
                    )

                    search_queries.append(final_query)

            if not search_queries:
              filter_snippets, filter_params = nvs.construct_metadata_filter(query.filters)
              index_query = base_index_query + filter_snippets + base_cosine_query

            else:
              if len(search_queries) == 1:
                index_query = search_queries[0]
              else:
                index_query = " UNION ".join(search_queries)

              index_query += "WITH node, max(score) AS score ORDER BY score DESC LIMIT $k "
              filter_params = {}

        else:
            index_query = nvs._get_search_index_query(self.hybrid_search)
            filter_params = {}

        default_retrieval = (
           f'''RETURN node.id AS name,
                [l in labels(node) WHERE NOT l IN ['{BASE_ENTITY_LABEL}', '{BASE_NODE_LABEL}'] | l][0] AS type,
                node{{.* , embedding: Null, name: Null, id: Null}} AS properties,
                score'''
        )

        retrieval_query = self.retrieval_query or default_retrieval
        read_query = index_query + retrieval_query
        parameters = {
            "index": self.index_name,
            "k": query.similarity_top_k,
            "embedding": query.query_embedding,
            "keyword_index": self.keyword_index_name,
            "query": nvs.remove_lucene_chars(query.query_str),
            **filter_params,
        }

        results = self.database_query(read_query, params=parameters)

        data = results if results else []

        nodes = []
        scores = []
        for record in data:
            node = EntityNode(
                name=record["name"],
                label=record["type"],
                properties=remove_empty_values(record["properties"]),
            )
            nodes.append(node)
            scores.append(record["score"])

        return (nodes, scores)

    def database_query(
        self,
        query: str,
        params: Optional[Dict[str, Any]] = None,
    ) -> Any:
        params = params or {}
        try:
            data, _, _ = self._driver.execute_query(
                query, database_=self.database, parameters_=params
            )
            return [r.data() for r in data]
        except neo4j.exceptions.Neo4jError as e:
            if not (
                (
                    (  # isCallInTransactionError
                        e.code == "Neo.DatabaseError.Statement.ExecutionFailed"
                        or e.code
                        == "Neo.DatabaseError.Transaction.TransactionStartFailed"
                    )
                    and "in an implicit transaction" in e.message
                )
                or (  # isPeriodicCommitError
                    e.code == "Neo.ClientError.Statement.SemanticError"
                    and (
                        "in an open transaction is not possible" in e.message
                        or "tried to execute in an explicit transaction" in e.message
                    )
                )
            ):
                raise
        # Fallback to allow implicit transactions
        with self._driver.session(database=self.database) as session:
            data = session.run(neo4j.Query(text=query), params)
            return [r.data() for r in data]

SyntaxError: invalid syntax. Perhaps you forgot a comma? (<ipython-input-6-5fdec58f9095>, line 62)

# Test MetaData Filtering with Hybrid Search (ye full lite)

In [ ]:

class Neo4jPropertyGraphStoreWithFTSAndMetadataFiltering(Neo4jPropertyGraphStoreWithFTS):
    def __init__(self, **kwargs: Any) -> None:
        super().__init__(**kwargs)
    def _get_search_index_query(self, hybrid: bool, filter_snippets: str = "") -> str:
        base_filter_query = (
                              f"MATCH (n:`{self.node_label}`) WHERE {filter_snippets} "
                              f"AND n.`{self.embedding_node_property}` IS NOT NULL "
                              if filter_snippets
                              else f"MATCH (n:`{self.node_label}`) WHERE n.`{self.embedding_node_property}` IS NOT NULL "
                            )

        if not hybrid:
            # Vector search only on filtered nodes
            return (
                f"{base_filter_query} "
                "WITH n "
                "CALL db.index.vector.queryNodes($index, $k, $embedding) "
                "YIELD node, score "
            )

        # Hybrid search (Vector + Keyword Search) on filtered nodes
        return (
            f"{base_filter_query} "
            "WITH n "
            "CALL { "
            "    CALL db.index.vector.queryNodes($index, $k, $embedding) "
            "    YIELD node, score "
            "    RETURN node, score "
            "    UNION ALL "
            "    CALL db.index.fulltext.queryNodes($keyword_index, $query, {limit: $k}) "
            "    YIELD node, score "
            "    RETURN node, score "
            "} "
            "WITH node, MAX(score) AS score "
            "ORDER BY score DESC LIMIT $k "
        )
    def vector_query(self, query: VectorStoreQuery, **kwargs: Any) -> Tuple[List[LabelledNode], List[float]]:
      filter_snippets, filter_params = "", {}

      if query.filters:
          # Construct metadata filters
          print("F")
          filter_snippets, filter_params = nvs.construct_metadata_filter(query.filters)
          print(filter_snippets)
          print(filter_params)

      # Construct index query with or without filters
      index_query = self._get_search_index_query(
          self.hybrid_search,
          filter_snippets=filter_snippets,
      )

      # Default retrieval query
      default_retrieval = (
          f'''RETURN node.id AS name,
              [l IN labels(node) WHERE NOT l IN ['{BASE_ENTITY_LABEL}', '{BASE_NODE_LABEL}'] | l][0] AS type,
              node{{.* , embedding: NULL, name: NULL, id: NULL}} AS properties,
              score'''
      )

      retrieval_query = self.retrieval_query or default_retrieval
      read_query = index_query + retrieval_query
      print(read_query)
      # Construct query parameters
      parameters = {
          "index": self.index_name,
          "k": query.similarity_top_k,
          "embedding": query.query_embedding,
          "keyword_index": self.keyword_index_name,
          "query": nvs.remove_lucene_chars(query.query_str),
          **filter_params,
      }

      # Execute query
      results = self.database_query(read_query, params=parameters)

      # Process results
      nodes = []
      scores = []
      for record in results:
          node = EntityNode(
              name=record["name"],
              label=record["type"],
              properties=remove_empty_values(record["properties"]),
          )
          nodes.append(node)
          scores.append(record["score"])

      return nodes, scores



# Custom Hybrid search Context Retriever


In [ ]:
import dataclasses
from typing import Any, List, Sequence, Optional, Dict, Set

from llama_index.core.base.embeddings.base import BaseEmbedding
from llama_index.core.indices.property_graph.sub_retrievers.vector import (
    VectorContextRetriever
)
from llama_index.core.graph_stores.types import (
    PropertyGraphStore,
    KG_SOURCE_REL,
    VECTOR_SOURCE_KEY,
)
from llama_index.core.settings import Settings
from llama_index.core.schema import BaseNode, NodeWithScore, QueryBundle, TextNode
from llama_index.core.vector_stores.types import (
    BasePydanticVectorStore,
    VectorStoreQuery,
    MetadataFilters,
)
class HybridContextRetriever(VectorContextRetriever):
    def __init__(
        self,
        graph_store: PropertyGraphStore,
        include_text: bool = True,
        include_properties: bool = False,
        embed_model: Optional[BaseEmbedding] = None,
        vector_store: Optional[BasePydanticVectorStore] = None,
        similarity_top_k: int = 4,
        path_depth: int = 1,
        limit: int = 30,
        similarity_score: Optional[float] = None,
        filters: Optional[MetadataFilters] = None,
        get_rel = False,
        **kwargs: Any,
    ) -> None:
        super().__init__(
            graph_store=graph_store,
            include_text=include_text,
            include_properties=include_properties,
            embed_model=embed_model,
            vector_store=vector_store,
            similarity_top_k=similarity_top_k,
            path_depth=path_depth,
            limit=limit,
            similarity_score=similarity_score,
            filters=filters,
            **kwargs,
        )
        self.get_rel = get_rel

    def _get_vector_store_query(self, query_bundle: QueryBundle) -> VectorStoreQuery:
        if query_bundle.embedding is None:
            query_bundle.embedding = self._embed_model.get_agg_embedding_from_queries(
                query_bundle.embedding_strs
            )
        # print(query_bundle)
        return VectorStoreQuery(
            query_str=query_bundle.query_str,
            query_embedding=query_bundle.embedding,
            similarity_top_k=self._similarity_top_k,
            filters=self._filters,
            **self._retriever_kwargs,
        )
    async def _aget_vector_store_query(
        self, query_bundle: QueryBundle
    ) -> VectorStoreQuery:
        if query_bundle.embedding is None:
            query_bundle.embedding = (
                await self._embed_model.aget_agg_embedding_from_queries(
                    query_bundle.embedding_strs
                )
            )

        return VectorStoreQuery(
            query_str=query_bundle.query_str,
            query_embedding=query_bundle.embedding,
            similarity_top_k=self._similarity_top_k,
            filters=self._filters,
            **self._retriever_kwargs,
        )

    def retrieve_from_graph(
        self, query_bundle: QueryBundle, limit: Optional[int] = None
    ) -> List[NodeWithScore]:
        vector_store_query = self._get_vector_store_query(query_bundle)

        triplets = []
        kg_ids = []
        new_scores = []
        if self._graph_store.supports_vector_queries:
            result = self._graph_store.vector_query(vector_store_query)
            if len(result) != 2:
                raise ValueError("No nodes returned by vector_query")
            kg_nodes, scores = result

            kg_ids = [node.id for node in kg_nodes]
            if self.get_rel:
              triplets = self._graph_store.get_rel_map(
                  kg_nodes,
                  depth=self._path_depth,
                  limit=limit or self._limit,
                  ignore_rels=[KG_SOURCE_REL],
              )

        elif self._vector_store is not None:
            query_result = self._vector_store.query(vector_store_query)
            if query_result.nodes is not None and query_result.similarities is not None:
                kg_ids = self._get_kg_ids(query_result.nodes)
                scores = query_result.similarities
                kg_nodes = self._graph_store.get(ids=kg_ids)
                if self.get_rel:
                  triplets = self._graph_store.get_rel_map(
                      kg_nodes,
                      depth=self._path_depth,
                      limit=limit or self._limit,
                      ignore_rels=[KG_SOURCE_REL],
                  )

            elif query_result.ids is not None and query_result.similarities is not None:
                kg_ids = query_result.ids
                scores = query_result.similarities
                kg_nodes = self._graph_store.get(ids=kg_ids)
                nodes = kg_nodes
                if self.get_rel:
                  triplets = self._graph_store.get_rel_map(
                      kg_nodes,
                      depth=self._path_depth,
                      limit=limit or self._limit,
                      ignore_rels=[KG_SOURCE_REL],
                  )

        if not self.get_rel:
          results = []

          for i, kg_node in enumerate(kg_nodes):
              if self.include_properties:
                  text = f"{kg_node!s}"
              else:
                  text = kg_node.id
              results.append(
                  NodeWithScore(
                      node=TextNode(
                          text=text,
                      ),
                      score=1.0 if scores is None else scores[i],
                  )
              )

          return results

        for triplet in triplets:
            score1 = (
                scores[kg_ids.index(triplet[0].id)] if triplet[0].id in kg_ids else 0.0
            )
            score2 = (
                scores[kg_ids.index(triplet[2].id)] if triplet[2].id in kg_ids else 0.0
            )
            new_scores.append(max(score1, score2))

        assert len(triplets) == len(new_scores)

        # filter by similarity score
        if self._similarity_score:
            filtered_data = [
                (triplet, score)
                for triplet, score in zip(triplets, new_scores)
                if score >= self._similarity_score
            ]
            # sort by score
            top_k = sorted(filtered_data, key=lambda x: x[1], reverse=True)
        else:
            # sort by score
            top_k = sorted(zip(triplets, new_scores), key=lambda x: x[1], reverse=True)
            scored_nodes = [(node, score) for node, score in zip(nodes, scores)]

    # Filter by similarity score if needed
        if self._similarity_score:
            filtered_data = [
                (node, score)
                for node, score in scored_nodes
                if score >= self._similarity_score
            ]
            # Sort by score
            top_k = sorted(filtered_data, key=lambda x: x[1], reverse=True)
        else:
            # Sort by score
            top_k = sorted(scored_nodes, key=lambda x: x[1], reverse=True)

        return self._get_nodes_with_score([x[0] for x in top_k], [x[1] for x in top_k])

    async def aretrieve_from_graph(
        self, query_bundle: QueryBundle, limit: Optional[int] = None
    ) -> List[NodeWithScore]:
        vector_store_query = await self._aget_vector_store_query(query_bundle)
        print(self._filters)
        triplets = []
        kg_ids = []
        new_scores = []
        if self._graph_store.supports_vector_queries:
            result = await self._graph_store.avector_query(vector_store_query)
            if len(result) != 2:
                raise ValueError("No nodes returned by vector_query")

            kg_nodes, scores = result
            kg_ids = [node.id for node in kg_nodes]
            if self.get_rel:
              triplets = await self._graph_store.aget_rel_map(
                  kg_nodes,
                  depth=self._path_depth,
                  limit=limit or self._limit,
                  ignore_rels=[KG_SOURCE_REL],
              )

        elif self._vector_store is not None:
            query_result = await self._vector_store.aquery(vector_store_query)
            if query_result.nodes is not None and query_result.similarities is not None:
                kg_ids = self._get_kg_ids(query_result.nodes)
                scores = query_result.similarities
                kg_nodes = await self._graph_store.aget(ids=kg_ids)
                if self.get_rel:
                  triplets = await self._graph_store.aget_rel_map(
                      kg_nodes,
                      depth=self._path_depth,
                      limit=limit or self._limit,
                      ignore_rels=[KG_SOURCE_REL],
                  )

            elif query_result.ids is not None and query_result.similarities is not None:
                kg_ids = query_result.ids
                scores = query_result.similarities
                kg_nodes = await self._graph_store.aget(ids=kg_ids)
                if self.get_rel:
                  triplets = await self._graph_store.aget_rel_map(
                      kg_nodes,
                      depth=self._path_depth,
                      limit=limit or self._limit,
                      ignore_rels=[KG_SOURCE_REL],
                  )

        if not self.get_rel:
          results = []

          for i, kg_node in enumerate(kg_nodes):
              if self.include_properties:
                  text = f"{kg_node!s}"
              else:
                  text = kg_node.id
              results.append(
                  NodeWithScore(
                      node=TextNode(
                          text=text,
                      ),
                      score=1.0 if scores is None else scores[i],
                  )
              )

          return results

        for triplet in triplets:
            score1 = (
                scores[kg_ids.index(triplet[0].id)] if triplet[0].id in kg_ids else 0.0
            )
            score2 = (
                scores[kg_ids.index(triplet[2].id)] if triplet[2].id in kg_ids else 0.0
            )
            new_scores.append(max(score1, score2))

        assert len(triplets) == len(new_scores)

        # filter by similarity score
        if self._similarity_score:
            filtered_data = [
                (triplet, score)
                for triplet, score in zip(triplets, new_scores)
                if score >= self._similarity_score
            ]
            # sort by score
            top_k = sorted(filtered_data, key=lambda x: x[1], reverse=True)
        else:
            # sort by score
            top_k = sorted(zip(triplets, new_scores), key=lambda x: x[1], reverse=True)

        return self._get_nodes_with_score([x[0] for x in top_k], [x[1] for x in top_k])

# COMStore (Code Object Model Store)

In [ ]:
from typing import Any, Dict, List, Optional, Tuple, Union
import logging
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.core.bridge.pydantic import PrivateAttr
from llama_index.core.vector_stores.types import (
    VectorStoreQuery
)
from llama_index.core.graph_stores.types import (
    LabelledNode
)
from llama_index.core.vector_stores.utils import (
    metadata_dict_to_node,
)
from llama_index.core import Settings

_logger = logging.getLogger(__name__)
BASE_ENTITY_LABEL = "__Entity__"
BASE_NODE_LABEL = "__Node__"

def check_if_not_null(props: List[str], values: List[Any]) -> None:
    """Check if variable is not null and raise error accordingly."""
    for prop, value in zip(props, values):
        if not value:
            raise ValueError(f"Parameter `{prop}` must not be None or empty string")

class COMStore(Neo4jPropertyGraphStore):
    stores_text: bool = True
    flat_metadata: bool = True
    distance_strategy: str
    index_name: str
    keyword_index_name: str
    hybrid_search: bool
    node_label: str
    embedding_node_property: str
    text_node_property: str
    retrieval_query: str
    _support_metadata_filter: bool = PrivateAttr()
    _is_enterprise: bool = PrivateAttr()
    custom_fts_indices: Any = []

    def __init__(
        self,
        username: str,
        password: str,
        url: str,
        database: Optional[str] = "neo4j",
        index_name: str = "entity",
        keyword_index_name: str = "keyword",
        node_label: str = "__Node__",
        embedding_node_property: str = "embedding",
        text_node_property: str = "source",
        retrieval_query: str = "",
        refresh_schema: bool = True,
        sanitize_query_output: bool = True,
        enhanced_schema: bool = False,
        **neo4j_kwargs: Any,
    ) -> None:
        self.index_name = index_name
        self.keyword_index_name = keyword_index_name
        self.node_label = node_label
        self.embedding_node_property = embedding_node_property
        self.text_node_property = text_node_property
        self.retrieval_query = retrieval_query
        self.database = database
        super().__init__(
            username,
            password,
            url,
            database,
            refresh_schema,
            sanitize_query_output,
            enhanced_schema,
            **neo4j_kwargs,
        )
        check_if_not_null(
            [
                "index_name",
                "node_label",
                "embedding_node_property",
                "text_node_property",
            ],
            [index_name, node_label, embedding_node_property, text_node_property],
        )
        self.create_required_indices()


    def create_required_indices(self) -> None:
      """
      This method constructs Cypher queries and executes them to create:
      1. Three new full-text indices in Neo4j:
        - one for MODULE entities on 'name' and 'source' properties.
        - one for CLASS entities on 'name' and 'source' properties.
        - one for METHOD entities on 'name' and 'source' properties.
      2. Three vector indices:
        - one for MODULE entities on the 'embedding' property.
        - one for CLASS entities on the 'embedding' property.
        - one for METHOD entities on the 'embedding' property.

      It uses the IF NOT EXISTS clause to avoid errors if the indices already exist.
      """

      # Create full-text indices
      module_index_query = """
      CREATE FULLTEXT INDEX module_index IF NOT EXISTS FOR (n:`MODULE`) ON EACH [n.`name`, n.`source`]
      """
      self.database_query(module_index_query)

      class_index_query = """
      CREATE FULLTEXT INDEX class_index IF NOT EXISTS FOR (n:`CLASS`) ON EACH [n.`name`, n.`source`]
      """
      self.database_query(class_index_query)

      method_index_query = """
      CREATE FULLTEXT INDEX method_index IF NOT EXISTS FOR (n:`METHOD`) ON EACH [n.`name`, n.`source`]
      """
      self.database_query(method_index_query)

      # Create vector indices
      module_vector_index_query = """
      CREATE VECTOR INDEX module_vector_index IF NOT EXISTS
      FOR (m:`MODULE`)
      ON (m.`embedding`)
      OPTIONS {indexConfig: {
        `vector.dimensions`: 1024,
        `vector.similarity_function`: 'cosine'
      }}
      """
      self.database_query(module_vector_index_query)

      class_vector_index_query = """
      CREATE VECTOR INDEX class_vector_index IF NOT EXISTS
      FOR (c:`CLASS`)
      ON (c.`embedding`)
      OPTIONS {indexConfig: {
        `vector.dimensions`: 1024,
        `vector.similarity_function`: 'cosine'
      }}
      """
      self.database_query(class_vector_index_query)

      method_vector_index_query = """
      CREATE VECTOR INDEX method_vector_index IF NOT EXISTS
      FOR (m:`METHOD`)
      ON (m.`embedding`)
      OPTIONS {indexConfig: {
        `vector.dimensions`: 1024,
        `vector.similarity_function`: 'cosine'
      }}
      """
      self.database_query(method_vector_index_query)

      general_fts_query = """
      CREATE FULLTEXT INDEX general_fts_index IF NOT EXISTS FOR (n:__Entity__) ON EACH [n.name, n.source]
      """
      self.database_query(general_fts_query)

    def query_vector_index(
        self, index_name: str, query: str | List[float],
        return_fields: Optional[List[str]] = None, **kwargs: Any
    ) -> List[Dict]:
        """
        Query a vector index.

        Args:
            index_name (str): The name of the vector index, restricted to `method_vector_index`, `class_vector_index` and `module_vector_index`.
            query (str or List[float]): The query or query vector embedding.
            return_fields (Optional[List[str]]): A list of fields to return.

        Returns:
            List[Dict]: A list of dictionaries containing the query results.
        """
        query_vector = None
        if isinstance(query, str):
            query_vector = Settings.embed_model.get_query_embedding(query)
        else:
            query_vector = query
        return_fields = return_fields or ["name", "source"]
        return_fields_str = ", ".join([f"n.{field} AS {field}" for field in return_fields])

        query_str = f"""
        CALL db.index.vector.queryNodes($index_name, $top_k, $query_vector)
        YIELD node AS n, score
        RETURN {return_fields_str}, score
        """
        query_params = {
            "index_name": index_name,
            "top_k": kwargs.get("top_k", 10),
            "query_vector": query_vector,
        }

        try:
            records = self.database_query(query_str, query_params)
            return records
        except Exception as e:
            _logger.error(f"Error querying vector index '{index_name}': {e}")
            return []

    def query_fulltext_index(
        self, index_name: str, query: str, return_fields: Optional[List[str]] = None,
        **kwargs: Any
    ) -> List[Dict]:
        """
        Query a full-text index.

        Args:
            index_name (str): The name of the full-text index, restricted to `module_index`, `class_index` and `method_index`.
            query (str): The query string.
            return_fields (Optional[List[str]]): A list of fields to return.

        Returns:
            List[Dict]: A list of dictionaries containing the query results.
        """
        return_fields = return_fields or ["name", "source"]
        return_fields_str = ", ".join([f"n.{field} AS {field}" for field in return_fields])

        query_str = f"""
        CALL db.index.fulltext.queryNodes($index_name, $query)
        YIELD node AS n, score
        RETURN {return_fields_str}, score
        """
        query_params = {
            "index_name": index_name,
            "query": query,
        }

        try:
            records = self.database_query(query_str, query_params)
            return records
        except Exception as e:
            _logger.error(f"Error querying full-text index '{index_name}': {e}")
            return []

    def similarity_search_without_index(
        self,
        labels: List[str],
        query: str | List[float],
        top_k: int = 10,
        return_fields: Optional[List[str]] = None,
        **kwargs: Any,
    ) -> List[Dict]:
        """
        Perform a similarity search without using an index.

        Args:
            labels (List[str]): A list of labels to query (e.g., ["MODULE", "CLASS", "METHOD"]).
            query (str or List[float]): The query or query vector embedding.
            top_k (int): The number of results to return. Defaults to 10.
            return_fields (Optional[List[str]]): A list of fields to return. Defaults to ["name", "source"]

        Returns:
            List[Dict]: A list of dictionaries containing the query results.
        """
        query_vector = None
        if isinstance(query, str):
            query_vector = Settings.embed_model.get_query_embedding(query)
        else:
            query_vector = query
        return_fields = return_fields or ["name", "source"]
        return_fields_str = ", ".join([f"n.{field} AS {field}" for field in return_fields])

        query_str = f"""
        MATCH (n)
        WHERE any(label in labels(n) WHERE label IN $labels)
        AND n.{self.embedding_node_property} IS NOT NULL
        WITH n,
        vector.similarity.cosine(n.{self.embedding_node_property}, $query_vector) AS similarity
        ORDER BY similarity DESC
        LIMIT $top_k
        RETURN {return_fields_str}, labels(n) AS all_labels, similarity AS score
        """

        query_params = {
            "labels": labels,
            "query_vector": query_vector,
            "top_k": top_k,
        }

        try:
            records = self.database_query(query_str, query_params)
            return records
        except Exception as e:
            _logger.error(f"Error performing similarity search: {e}")
            return []

    def database_query(
        self,
        query: str,
        params: Optional[Dict[str, Any]] = None,
    ) -> Any:
        """
        Execute a cypher query in the Neo4j database.

        Args:
            query (str): The query string.
            params (Optional[Dict[str, Any]]): A dictionary of parameters to pass to the query.
        """
        params = params or {}
        try:
            data, _, _ = self._driver.execute_query(
                query, database_=self.database, parameters_=params
            )
            return [r.data() for r in data]
        except neo4j.exceptions.Neo4jError as e:
            if not (
                (
                    (  # isCallInTransactionError
                        e.code == "Neo.DatabaseError.Statement.ExecutionFailed"
                        or e.code
                        == "Neo.DatabaseError.Transaction.TransactionStartFailed"
                    )
                    and "in an implicit transaction" in e.message
                )
                or (  # isPeriodicCommitError
                    e.code == "Neo.ClientError.Statement.SemanticError"
                    and (
                        "in an open transaction is not possible" in e.message
                        or "tried to execute in an explicit transaction" in e.message
                    )
                )
            ):
                raise
        # Fallback to allow implicit transactions
        with self._driver.session(database=self.database) as session:
            data = session.run(neo4j.Query(text=query), params)
            return [r.data() for r in data]

# Import required modules and initalize settings

In [ ]:
import nest_asyncio
import asyncio

nest_asyncio.apply()

In [ ]:
import sys
import logging

logging.basicConfig(stream=sys.stdout, level=logging.ERROR)
logging.getLogger().addHandler(logging.StreamHandler(sys.stdout))

In [ ]:
import inspect
import importlib
import json
from typing import List
from llama_index.core import Settings
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.core import PropertyGraphIndex
from llama_index.core.graph_stores.types import EntityNode, Relation
from llama_index.llms.gemini import Gemini
from llama_index.embeddings.gemini import GeminiEmbedding
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.schema import MetadataMode
from google.colab import userdata

# Initialize Neo4j connection
pg_store =  COMStore(
    username="neo4j",
    password=userdata.get("NEO4J_PASSWORD_KG"),
    url=userdata.get("NEO4J_URI_KG"),
    # hybrid_search=False
)

# Refine this schema later
schema = [
    ("MODULE", "HAS_CLASS", "CLASS"),
    ("MODULE", "HAS_CONSTANT", "CONSTANT"),
    ("CLASS", "HAS_BASECLASS", "CLASS"),
    ("CLASS", "HAS_PARAMETER", "PARAMETER"),
    ("CLASS", "HAS_METHOD", "METHOD"),
    ("METHOD", "HAS_PARAMETER", "PARAMETER"),
    ("METHOD", "HAS_RETURN_TYPE", "RETURN_TYPE"),
    ("PARAMETER", "HAS_TYPE", "TYPE"),
    ("PARAMETER", "HAS_DEFAULT_VALUE", "DEFAULT_VALUE"),
    ("CONSTANT", "HAS_VALUE", "VALUE"),
]

## Create GeminiWithRateLimits class

In [ ]:
from typing import Sequence
from llama_index.llms.gemini.utils import (
    chat_from_gemini_response,
    chat_message_to_gemini,
    completion_from_gemini_response,
)
from llama_index.core.base.llms.types import (
    ChatMessage,
    ChatResponse,
)
from llama_index.core.bridge.pydantic import Field
from llama_index.core.constants import DEFAULT_NUM_OUTPUTS, DEFAULT_TEMPERATURE
from llama_index.core.callbacks import CallbackManager
from ratelimit import limits, sleep_and_retry
import google.generativeai as genai

GEMINI_MODELS = (
    "models/gemini-2.0-flash-exp",
    # Gemini 1.0 Pro Vision has been deprecated on July 12, 2024.
    # According to official recommendations, switch the default model to gemini-1.5-flash
    "models/gemini-1.5-flash",
    "models/gemini-1.5-flash-latest",
    "models/gemini-pro",
    "models/gemini-pro-latest",
    "models/gemini-1.5-pro",
    "models/gemini-1.5-pro-latest",
    "models/gemini-1.0-pro",
    # for some reason, google lists this without the models prefix
    "gemini-1.5-flash",
    "gemini-1.5-flash-latest",
    "gemini-1.0-pro",
)
from llama_index.core.utilities.gemini_utils import (
    merge_neighboring_same_role_messages,
)

class GeminiWithRateLimits(Gemini):
    rate_limit: int = Field(
        default=10,
        # description=""
    )
    period: int = Field(
        default=60,
        # description=""
    )
    def __init__(
        self,
        api_key: Optional[str] = None,
        model: str = GEMINI_MODELS[0],
        temperature: float = DEFAULT_TEMPERATURE,
        max_tokens: Optional[int] = None,
        generation_config: Optional[genai.types.GenerationConfigDict] = None,
        safety_settings: Optional[genai.types.SafetySettingDict] = None,
        callback_manager: Optional[CallbackManager] = None,
        api_base: Optional[str] = None,
        transport: Optional[str] = None,
        model_name: Optional[str] = None,
        default_headers: Optional[Dict[str, str]] = None,
        request_options: Optional[genai.types.RequestOptions] = None,
        rate_limit: Optional[int] = None,
        period: Optional[int] = None,
        **generate_kwargs: Any,
    ):
        """Creates a new Gemini model interface."""
        super().__init__(
            api_key=api_key,
            model=model,
            temperature=temperature,
            max_tokens=max_tokens,
            generation_config=generation_config,
            safety_settings=safety_settings,
            callback_manager=callback_manager,
            api_base=api_base,
            transport=transport,
            model_name=model_name,
            default_headers=default_headers,
            request_options=request_options,
            **generate_kwargs,
        )
        self.rate_limit = rate_limit or 10
        self.period = period or 60

    def _get_limiter(self):
      """Gets the rate limiter for this instance."""
      return limits(calls=self.rate_limit, period=self.period)

    async def achat(
        self, messages: Sequence[ChatMessage], **kwargs: Any
    ) -> ChatResponse:
        limiter = self._get_limiter()
        request_options = self._request_options or kwargs.pop("request_options", None)
        merged_messages = merge_neighboring_same_role_messages(messages)
        *history, next_msg = map(chat_message_to_gemini, merged_messages)
        chat = self._model.start_chat(history=history)

        @sleep_and_retry
        @limiter
        async def limited_send_message():
            return await chat.send_message_async(
                next_msg, request_options=request_options, **kwargs
            )

        response = await limited_send_message()
        return chat_from_gemini_response(response)

In [ ]:
Settings.llm = GeminiWithRateLimits(
    api_key = userdata.get("GOOGLE_API_KEY"),
    temperature=0.01,
    rate_limit=10,
    period=60
)

In [ ]:
Settings.llm

GeminiWithRateLimits(callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x7b0296eacfd0>, system_prompt=None, messages_to_prompt=<function messages_to_prompt at 0x7b03aa1007c0>, completion_to_prompt=<function default_completion_to_prompt at 0x7b03a9f739c0>, output_parser=None, pydantic_program_mode=<PydanticProgramMode.DEFAULT: 'default'>, query_wrapper_prompt=None, model='models/gemini-2.0-flash-exp', temperature=0.01, max_tokens=8192, generate_kwargs={}, rate_limit=10, period=60)

In [ ]:
# Use for Gemini Embeddings
# Settings.embed_model = GeminiEmbedding(
#     embed_batch_size = 25,
#     model_name = "models/text-embedding-004",
#     api_key = userdata.get("GOOGLE_API_KEY")
# )

In [ ]:
# Use for HuggingFace Embeddings
Settings.embed_model = HuggingFaceEmbedding(
    model_name = "mixedbread-ai/mxbai-embed-large-v1",
    embed_batch_size = 25,
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/114k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/670M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [ ]:
print(Settings.embed_model)

model_name='mixedbread-ai/mxbai-embed-large-v1' embed_batch_size=25 callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x7cab74410eb0> num_workers=None max_length=512 normalize=True query_instruction=None text_instruction=None cache_folder=None


# Parse docs to get the user-level module stuff only

In [ ]:
import asyncio
import re
from typing import List, Optional, Dict, Set

from llama_index.readers.web import BeautifulSoupWebReader
from llama_index.tools.duckduckgo import DuckDuckGoSearchToolSpec
from ratelimit import limits, sleep_and_retry
from llama_index.core.prompts import PromptTemplate
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.llms import ChatMessage
from llama_index.core import Settings
from collections import defaultdict
from bs4 import BeautifulSoup
import requests
from urllib.parse import urljoin

ddgs = DuckDuckGoSearchToolSpec()

RATE_LIMIT = 9  # requests per minute

@sleep_and_retry
@limits(calls=RATE_LIMIT, period=60)  # 9 calls per minute
async def call_model(messages: List[ChatMessage]):
    return await Settings.llm.achat(messages)

async def extract_structure_from_page(
    url: str, nodes, prompt_template, llm
) -> Dict[str, Set[str]]:
    """
    Extracts the structure from the page by using LLM.
    Processes each node from the text splitter and aggregates the results into a dictionary of sets.
    """
    results: Dict[str, Set[str]] = defaultdict(set)

    messages = [
        ChatMessage(
            role="system",
            content=prompt_template.template,
        )
    ]

    for node in nodes:
        messages.append(ChatMessage(role="user", content=node.get_content()))

        response = await call_model(messages)

        entities_by_type = parse_llm_response(response.message.content)

        for entity_type, entities in entities_by_type.items():
            results[entity_type].update(entities)  # Update the sets

        messages.append(response.message)

    return results

def parse_llm_response(response_content: str) -> Dict[str, Set[str]]:
    """
    Parses the LLM response content and extracts entities into a dictionary of sets based on their type.
    """
    entities_by_type: Dict[str, Set[str]] = defaultdict(set)
    lines = response_content.split("\n")

    for line in lines:
        line = line.strip()
        if not line:
            continue

        parts = line.split(":", 1)  # Split on the first colon only
        if len(parts) == 2:
            entity_type = parts[0].strip().lower()
            entity = parts[1].strip()

            if entity_type in ["module", "class"]:
                entities_by_type[entity_type].add(entity)

    return entities_by_type

# Define the function to recursively parse the website
async def recursive_parse_website(
    start_url: str,
    base_path: str,
    max_depth: int | None,
    llm,
    visited_urls=None,
    current_depth=0,
) -> Dict[str, Set[str]]:
    """Recursively parses a website and extracts information."""

    if visited_urls is None:
        visited_urls = set()

    if (
        max_depth is not None
        and current_depth > max_depth
        or start_url in visited_urls
    ):
        return defaultdict(set)

    visited_urls.add(start_url)
    print(f"Processing {start_url} at depth {current_depth}")
    results: Dict[str, Set[str]] = defaultdict(set)

    prompt_template = PromptTemplate(
        """
        You are tasked with extracting structured information from Python module documentation.
        Analyze the given content and identify all the classes and modules, along with their full paths.
        If an entity is clearly a class or module, categorize it as 'class' or 'module' respectively.
        If there is any accompanying code available, please use that to identify the entities.
        The entities may not be displayed with their full path. Smartly find the full paths for the entities in these cases.
        Return each entity on a new line, prefixed with its type followed by a colon, and then the entity's full path or name.

        Example:
        class: module.submodule.ClassName
        module: module

        Page Content:
        {text}
        """
    )

    try:
        response = requests.get(start_url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")

        loader = BeautifulSoupWebReader()
        documents = loader.load_data(urls=[start_url])

        if not documents:
            return defaultdict(set)

        chunk_size = Settings.llm.max_tokens

        splitter = SentenceSplitter(
            chunk_size=chunk_size,
            chunk_overlap=int(0.02 * chunk_size),
        )
        nodes = splitter.get_nodes_from_documents(documents)

        extracted_info = await extract_structure_from_page(
            start_url, nodes, prompt_template, llm
        )
        results: Dict[str, Set[str]] = defaultdict(set, extracted_info)

        links = set()
        for link in soup.find_all("a", href=True):
            href = link.get("href")
            if "#" in href:
              href = href.split("#")[0]  # Remove anchor part
            if not href:
              continue

            absolute_url = urljoin(start_url, href)  # Construct absolute URL

            if absolute_url.startswith(base_path):
                links.add(absolute_url)

        # Recursively process links
        sub_results = await asyncio.gather(
            *[
                recursive_parse_website(
                    url, base_path, max_depth, llm, visited_urls, current_depth + 1
                )
                for url in links
            ]
        )

        for sub_result in sub_results:
            for key, value in sub_result.items():
                results[key].update(value)

    except Exception as e:
        print(f"Could not get links from page {start_url}: {e}")
        return results

    return results

async def get_api_reference_for_module(
    module_name: str, llm
) -> tuple[str, str]:
    """
    Finds the API reference for a given Python module using DuckDuckGo search,
    selects the most relevant link using the LLM, and then determines the base path
    for recursive parsing. Returns both the most relevant URL and the base path.
    """

    query = f"{module_name} Python API Reference"
    search_results = ddgs.duckduckgo_full_search(query=query)

    # Construct a prompt for the LLM to select the best link
    search_results_text = ""
    for i, result in enumerate(search_results):
        search_results_text += f"{i+1}. {result['title']} ({result['href']})\n"

    prompt_template = PromptTemplate(
        """
        You are given a list of search results for the query: "{query}".
        Please select the URL that is most likely to contain the API reference documentation
        for the Python module.

        Additionally, analyze the selected URL and determine a suitable base path for a website crawler.
        The base path maybe a prefix of the selected URL or the selected URL itself that points to a section
        of the website likely to contain API reference or documentation pages. Consider common URL structures
        and keywords like 'api', 'api_reference', 'reference', 'apireference', or 'docs'.

        Keep in mind that you may return the selected URL as the base path if it looks like a valid base path.

        Return the output in the format:
        <selected_url>
        <base_path>

        Search Results:
        {search_results_text}

        Most relevant URL and base path:
        """
    )

    # Get the LLM's selection
    response = (
        await llm.apredict(
            prompt_template, query=query, search_results_text=search_results_text
        )
    ).strip()
    try:
        selected_url, base_path = response.split("\n")
    except ValueError:
        print(f"Could not get url and base path from llm response: {response}")
        return "", ""

    return selected_url, base_path

async def extract_user_level_functionalities_for_module(
    module_name: str, llm, max_depth: Optional[int] = 2
) -> Dict[str, Set[str]]:
    """
    This is a wrapper function that finds the user level functionalities for a given module by:
    1. Finding the API reference URL and base path using `get_api_reference_for_module`.
    2. Recursively parsing the website using `recursive_parse_website`.

    Returns:
        A dictionary of sets, where keys are entity types (module, class, method/function) and values are sets of corresponding entities.
    """
    selected_url, base_path = await get_api_reference_for_module(module_name, llm)

    if not selected_url or not base_path:
        print(f"Could not find API reference for module: {module_name}")
        return defaultdict(set)

    print(f"Extracting information from: {selected_url} (Base path: {base_path})")

    results = await recursive_parse_website(
        selected_url, base_path, max_depth, llm
    )

    return results

In [ ]:
#@title Example Usage of DDGS for Finding API Reference links for Python modules
ddgs.duckduckgo_full_search("pydantic Python API Reference")

[{'title': 'Welcome to Pydantic - Pydantic',
  'href': 'https://docs.pydantic.dev/latest/',
  'body': 'Pydantic¶ Documentation for version: v2.10.5. Pydantic is the most widely used data validation library for Python. Fast and extensible, Pydantic plays nicely with your linters/IDE/brain. Define how data should be in pure, canonical Python 3.8+; validate it with Pydantic.'},
 {'title': 'Web and API Requests - Pydantic',
  'href': 'https://docs.pydantic.dev/latest/examples/requests/',
  'body': "Pydantic is instrumental in many web frameworks and libraries, such as FastAPI, Django, Flask, and HTTPX. httpx requests¶ httpx is a HTTP client for Python 3 with synchronous and asynchronous APIs. In the below example, we query the JSONPlaceholder API to get a user's data and validate it with a Pydantic model."},
 {'title': 'python - FastAPI / Pydantic circular references in separate files ...',
  'href': 'https://stackoverflow.com/questions/63420889/fastapi-pydantic-circular-references-in-sepa

In [ ]:
#@title Example Usage of get_api_reference_for_module
module = "requests"
url, base_path = await get_api_reference_for_module(module, llm = Settings.llm)
print(f"URL: {url}")
print(f"Base Path: {base_path}")

URL: https://docs.python-requests.org/en/latest/api/
Base Path: https://docs.python-requests.org/en/latest/api/


In [ ]:
module_name = "requests"  # Example module
max_depth = None  # You can adjust the depth

extracted_info = await extract_user_level_functionalities_for_module(
    module_name, Settings.llm, max_depth
)

Extracting information from: https://docs.python-requests.org/en/latest/api/ (Base path: https://docs.python-requests.org/en/latest/api/)
Processing https://docs.python-requests.org/en/latest/api/ at depth 0


In [ ]:
print("Number of extracted modules:", len(extracted_info["module"]))
print("Number of extracted classes:", len(extracted_info["class"]))

Number of extracted modules: 5
Number of extracted classes: 21


In [ ]:
extracted_info["module"]

{'requests',
 'requests.adapters',
 'requests.auth',
 'requests.cookies',
 'requests.utils'}

Verify the above results like so:


1.   Head over to the [requests API Reference](https://docs.python-requests.org/en/latest/api/).
2.   Use the following code to find the number of methods/functions: (DEPRECATED)


```javascript
const elements = document.querySelectorAll('.py.method, .py.function');

// Convert the NodeList to an array
const elementsArray = Array.from(elements);
console.log(elementsArray.length); // 94
```

3.   Use the following code to find the number of classes (exceptions included):


```javascript
const elements = document.querySelectorAll('.py.class, .py.exception');

// Convert the NodeList to an array
const elementsArray = Array.from(elements);
console.log(elementsArray.length); // 21
```






## Testing extract_user_level_functionalities_for_module with pydantic

In [ ]:
module_name = "pydantic"  # Example module
max_depth = None  # You can adjust the depth

extracted_info = await extract_user_level_functionalities_for_module(
    module_name, Settings.llm, max_depth
)

Extracting information from: https://docs.pydantic.dev/latest/api/base_model/ (Base path: https://docs.pydantic.dev/latest/api/)
Processing https://docs.pydantic.dev/latest/api/base_model/ at depth 0
Processing https://docs.pydantic.dev/latest/api/functional_serializers/ at depth 1
Processing https://docs.pydantic.dev/latest/api/config/ at depth 1
Processing https://docs.pydantic.dev/latest/api/functional_validators/ at depth 1
Processing https://docs.pydantic.dev/latest/api/pydantic_extra_types_color/ at depth 1
Processing https://docs.pydantic.dev/latest/api/pydantic_extra_types_phone_numbers/ at depth 1
Processing https://docs.pydantic.dev/latest/api/pydantic_extra_types_script_code/ at depth 1
Processing https://docs.pydantic.dev/latest/api/pydantic_extra_types_pendulum_dt/ at depth 1
Processing https://docs.pydantic.dev/latest/api/pydantic_settings/ at depth 1
Processing https://docs.pydantic.dev/latest/api/json_schema/ at depth 1
Processing https://docs.pydantic.dev/latest/api/ex

In [ ]:
print("Number of extracted modules:", len(extracted_info["module"]))
print("Number of extracted classes:", len(extracted_info["class"]))
# print("Number of extracted methods/functions:", len(extracted_info["method/function"]))

Number of extracted modules: 109
Number of extracted classes: 384


## Testing extract_user_level_functionalities_for_module with bs4

In [ ]:
module_name = "BeautifulSoup4"  # Example module
max_depth = None  # You can adjust the depth

extracted_info = await extract_user_level_functionalities_for_module(
    module_name, Settings.llm, max_depth
)

Extracting information from: https://www.crummy.com/software/BeautifulSoup/bs4/doc/ (Base path: https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
Processing https://www.crummy.com/software/BeautifulSoup/bs4/doc/ at depth 0
Processing https://www.crummy.com/software/BeautifulSoup/bs4/doc/py-modindex.html at depth 1
Processing https://www.crummy.com/software/BeautifulSoup/bs4/doc/genindex.html at depth 1
Processing https://www.crummy.com/software/BeautifulSoup/bs4/doc/_sources/index.rst.txt at depth 1
Processing https://www.crummy.com/software/BeautifulSoup/bs4/doc/index.html at depth 2


Could not get links from page https://www.crummy.com/software/BeautifulSoup/bs4/doc/index.html: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-exp:generateContent?%24alt=json%3Benum-encoding%3Dint: Resource has been exhausted (e.g. check quota).
429 POST /v1beta/models/gemini-2.0-flash-exp:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1597.20ms


In [ ]:
print("Number of extracted modules:", len(extracted_info["module"]))
print("Number of extracted classes:", len(extracted_info["class"]))
# print("Number of extracted methods/functions:", len(extracted_info["method/function"]))

Number of extracted modules: 1
Number of extracted classes: 16


# Create necessary functions

In [ ]:
# Helper to create entity nodes and relations
def create_entity(label: str, name: str, properties: dict = None) -> EntityNode:
    # Convert properties to strings to ensure serializability
    properties = {k: str(v) for k, v in (properties or {}).items()}
    return EntityNode(label=label, name=name, properties=properties)

def create_relation(label: str, source: EntityNode, target: EntityNode, properties: dict = None) -> Relation:
    # Convert properties to strings to ensure serializability
    properties = {k: str(v) for k, v in (properties or {}).items()}
    return Relation(label=label, source_id=source.id, target_id=target.id, properties=properties)

# Upsert entities and relations into Neo4j
def upsert_to_neo4j(entities: List[EntityNode], relations: List[Relation]):
    pg_store.upsert_nodes(entities)
    pg_store.upsert_relations(relations)

def add_labels(module_name):
    # Step 1: Label modules recursively
    module_query = (
        """
        MATCH (base:MODULE {name: $module_name})-[*0..]->(descendant:MODULE)
        WITH DISTINCT descendant.name AS moduleNames
        UNWIND moduleNames AS moduleName
        MATCH (parent:MODULE {name: moduleName})-[*0..]->(descendant)
        CALL apoc.create.addLabels(parent, [moduleName]) YIELD node as pnode
        CALL apoc.create.addLabels(descendant, [moduleName]) YIELD node as dnode
        RETURN moduleName, COUNT(DISTINCT descendant) AS TotalNodesLabeled
        """
    )
    pg_store.structured_query(module_query, {"module_name": module_name})

    # Step 2: Label classes and their subtrees recursively
    class_query = (
        """
        MATCH (base:MODULE {name: $module_name})-[:HAS_CLASS]->(class:CLASS)
        WITH DISTINCT class
        MATCH (class)-[*0..]->(descendant)
        CALL apoc.create.addLabels(class, [class.name]) YIELD node as cnode
        CALL apoc.create.addLabels(descendant, [class.name]) YIELD node as dnode
        RETURN class.name AS ClassName, COUNT(DISTINCT descendant) AS TotalNodesLabeled
        """
    )
    pg_store.structured_query(class_query, {"module_name": module_name})


def create_module_fts_index(module_name):
    pg_store.create_new_keyword_index(node_label = module_name, index_name = module_name)
    pg_store.custom_fts_indices_names.append(module_name)

In [ ]:
#@title Someone fix this

import importlib
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import inspect
import subprocess

def fetch_pypi_documentation_and_homepage_urls(module_name: str):
    """
    Fetches both the 'Homepage' and 'Documentation' URLs for a module from PyPI.
    """
    pypi_url = f"https://pypi.org/project/{module_name}/"
    urls = {"Homepage": None, "Documentation": None}  # Default values

    try:
        response = requests.get(pypi_url)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # Find links labeled 'Homepage' and 'Documentation'
        for link_text in urls.keys():
            link = soup.find("a", string=link_text)
            if link:
                urls[link_text] = link['href']  # Store the URL in the dictionary
    except Exception as e:
        print(f"Error fetching PyPI page for {module_name}: {e}")

    return urls

def parse_documentation_for_relevant_entities(doc_url: str, visited_urls=None):
    """
    Parses the documentation at the provided URL to extract relevant entities (methods, modules, classes, constants).
    Recursively processes documentation links within the same domain.
    """
    import re

    if visited_urls is None:
        visited_urls = set()

    # Avoid visiting the same URL multiple times
    if doc_url in visited_urls:
        return set()
    visited_urls.add(doc_url)

    print(f"Visiting URL: {doc_url}")

    try:
        response = requests.get(doc_url)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        relevant_entities = set()

        # Regex patterns to identify relevant entities
        method_pattern = re.compile(r"^\w+\(.*\)$")  # Matches something like `method_name(param)`
        class_pattern = re.compile(r"^[A-Z][a-zA-Z0-9]+$")  # Matches PascalCase class names
        constant_pattern = re.compile(r"^[A-Z0-9_]+$")  # Matches ALL_CAPS constants
        module_pattern = re.compile(r"^\w+(\.\w+)*$")  # Matches module names (e.g., `module.submodule`)

        # Extract code blocks
        for code_block in soup.find_all("code"):
            code_text = code_block.text.strip()

            # Match against relevant patterns
            if method_pattern.match(code_text):
                print("Method:", code_text)
                relevant_entities.add(code_text)
            elif class_pattern.match(code_text):
                print("Class:", code_text)
                relevant_entities.add(code_text)
            elif constant_pattern.match(code_text):
                print("Constant:", code_text)
                relevant_entities.add(code_text)
            elif module_pattern.match(code_text):
                print("Module:", code_text)
                relevant_entities.add(code_text)

        # Extract links that might point to other documentation pages
        for link in soup.find_all("a"):
            href = link.get("href")
            if href:
                full_url = urljoin(doc_url, href)

                # Limit crawling to links within the same domain and avoid redundant crawling
                if full_url.startswith(doc_url) and full_url not in visited_urls:
                    if any(keyword in href.lower() for keyword in ["class", "method", "function", "api", "module"]):
                        # Recursively parse documentation links
                        relevant_entities.update(parse_documentation_for_relevant_entities(full_url, visited_urls))

        return relevant_entities

    except Exception as e:
        print(f"Error parsing documentation at {doc_url}: {e}")
        return set()

def traverse_module_recursive(module, parent_entity=None, full_path="", relevant_entities=None):
    """
    Recursively traverse a module, capturing its structure, relationships, and properties, including constants,
    but limiting to relevant entities if provided.
    """
    entities = []
    relations = []

    module_name = module.__name__
    current_path = f"{full_path}.{module_name}" if full_path else module_name

    # MODULE entity
    module_entity = create_entity(
        "MODULE",
        current_path,
        properties={"doc": inspect.getdoc(module), "comments": inspect.getcomments(module)},
    )
    entities.append(module_entity)

    # If parent entity is provided, create HAS_MODULE relation
    if parent_entity:
        relations.append(create_relation("HAS_MODULE", parent_entity, module_entity))

    # Traverse module attributes
    for name, obj in inspect.getmembers(module):
        obj_full_path = f"{current_path}.{name}"

        # Skip irrelevant entities if relevant_entities is specified
        if relevant_entities and name not in relevant_entities:
            continue

        if inspect.ismodule(obj):
            # Skip built-in or external modules unless part of the same root
            if obj.__name__.startswith(module.__name__):
                sub_entities, sub_relations = traverse_module_recursive(
                    obj, module_entity, current_path, relevant_entities
                )
                entities.extend(sub_entities)
                relations.extend(sub_relations)

        elif inspect.isclass(obj):
            # CLASS entity
            class_entity = create_entity(
                "CLASS",
                obj_full_path,
                properties={"doc": inspect.getdoc(obj), "comments": inspect.getcomments(obj), "isAbstract": inspect.isabstract(obj)},
            )
            entities.append(class_entity)
            relations.append(create_relation("HAS_CLASS", module_entity, class_entity))

            # Class methods and base classes
            for method_name, method_obj in inspect.getmembers(obj, predicate=inspect.isfunction):
                method_full_path = f"{obj_full_path}.{method_name}"

                # Collect method parameters as a list of dictionaries
                params = inspect.signature(method_obj).parameters
                params_list = [
                    {
                        "name": param_name,
                        "type": str(param.annotation) if param.annotation != inspect.Parameter.empty else "Any",
                        "default": param.default if param.default != inspect.Parameter.empty else None,
                        "kind": str(param.kind)
                    }
                    for param_name, param in params.items()
                ]

                method_entity = create_entity(
                    "METHOD",
                    method_full_path,
                    properties={
                        "comments": inspect.getcomments(method_obj),
                        "doc": inspect.getdoc(method_obj),
                        "signature": str(inspect.signature(method_obj)),
                        "source": inspect.getsource(method_obj),
                        "parameters": params_list,  # Store parameters in the method properties
                    },
                )
                entities.append(method_entity)
                relations.append(create_relation("HAS_METHOD", class_entity, method_entity))

            # Base classes
            for base_class in obj.__bases__:
                base_class_entity = create_entity("CLASS", base_class.__name__)
                entities.append(base_class_entity)
                relations.append(create_relation("HAS_BASECLASS", class_entity, base_class_entity))

        elif inspect.isfunction(obj):
            # MODULE-LEVEL FUNCTION entity
            params = inspect.signature(obj).parameters
            params_list = [
                {
                    "name": param_name,
                    "type": str(param.annotation) if param.annotation != inspect.Parameter.empty else "Any",
                    "default": param.default if param.default != inspect.Parameter.empty else None,
                    "kind": str(param.kind)
                }
                for param_name, param in params.items()
            ]

            function_entity = create_entity(
                "METHOD",
                obj_full_path,
                properties={
                    "comments": inspect.getcomments(obj),
                    "doc": inspect.getdoc(obj),
                    "signature": str(inspect.signature(obj)),
                    "source": inspect.getsource(obj),
                    "parameters": params_list,  # Store parameters in the function properties
                },
            )
            entities.append(function_entity)
            relations.append(create_relation("HAS_METHOD", module_entity, function_entity))

        elif not name.startswith("__"):  # Ignore dunder attributes
            # Handle constants or other module-level attributes
            if not inspect.isroutine(obj) and not inspect.isclass(obj) and not inspect.ismodule(obj):
                constant_entity = create_entity(
                    "CONSTANT",
                    obj_full_path,
                    properties={
                        "value": repr(obj),
                        "type": type(obj).__name__,
                    },
                )
                entities.append(constant_entity)
                relations.append(create_relation("HAS_CONSTANT", module_entity, constant_entity))

    return entities, relations

def install_and_import(module_name: str):
    """
    Ensures a module is installed and imports it.
    """
    try:
        return importlib.import_module(module_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", module_name])
        return importlib.import_module(module_name)

def process_module_recursive(module_name: str):
    """
    Processes a module by fetching documentation, parsing it for relevance,
    and then traversing its structure based on relevant entities.
    """
    urls = fetch_pypi_documentation_and_homepage_urls(module_name)

    print("URLs fetched:", urls["Homepage"], ",", urls["Documentation"])

    relevant_entities = set()
    if urls["Homepage"]:
        relevant_entities.union(parse_documentation_for_relevant_entities(urls["Homepage"]))
    if urls["Documentation"]:
        relevant_entities.union(parse_documentation_for_relevant_entities(urls["Documentation"]))

    # Import module
    module = install_and_import(module_name)

    # Traverse module structure based on relevant entities
    entities, relations = traverse_module_recursive(module, relevant_entities=relevant_entities)
    upsert_to_neo4j(entities, relations)

In [ ]:
#@title This code somewhat works
def traverse_module_recursive(module_or_object, parent_entity=None, full_path=""):
    """
    Recursively traverse a module, capturing its structure, relationships, and properties, including constants.
    Handles cases where input may be a class or object from an external module.
    """
    entities = []
    relations = []

    if inspect.ismodule(module_or_object):
        # Handle module as root
        module_name = module_or_object.__name__
        current_path = f"{full_path}.{module_name}" if full_path else module_name

        # MODULE entity
        module_entity = create_entity(
            "MODULE",
            current_path,
            properties={"doc": inspect.getdoc(module_or_object), "comments": inspect.getcomments(module_or_object)},
        )
        entities.append(module_entity)

        # If parent entity is provided, create HAS_MODULE relation
        if parent_entity:
            relations.append(create_relation("HAS_MODULE", parent_entity, module_entity))

        # Traverse module attributes
        for name, obj in inspect.getmembers(module_or_object):
            obj_full_path = f"{current_path}.{name}"
            # Recur for module members
            entities, relations = process_member(
                name, obj, module_entity, current_path, entities, relations
            )

    elif inspect.isclass(module_or_object):
        # Handle external class directly as root
        class_name = module_or_object.__name__
        current_path = full_path or class_name

        class_entity = create_entity(
            "CLASS",
            current_path,
            properties={
                "doc": inspect.getdoc(module_or_object),
                "comments": inspect.getcomments(module_or_object),
                "isAbstract": inspect.isabstract(module_or_object),
            },
        )
        entities.append(class_entity)

        # Traverse class methods and base classes
        for name, obj in inspect.getmembers(module_or_object):
            obj_full_path = f"{current_path}.{name}"
            entities, relations = process_member(
                name, obj, class_entity, current_path, entities, relations
            )

    return entities, relations


def process_member(name, obj, parent_entity, current_path, entities, relations):
    """
    Helper function to process a single member (class, function, constant).
    """
    obj_full_path = f"{current_path}.{name}"

    if inspect.ismodule(obj):
        # Skip built-in or unrelated external modules
        if obj.__name__.startswith(parent_entity["id"]):
            sub_entities, sub_relations = traverse_module_recursive(obj, parent_entity, current_path)
            entities.extend(sub_entities)
            relations.extend(sub_relations)

    elif inspect.isclass(obj):
        # CLASS entity
        class_entity = create_entity(
            "CLASS",
            obj_full_path,
            properties={
                "doc": inspect.getdoc(obj),
                "comments": inspect.getcomments(obj),
                "isAbstract": inspect.isabstract(obj),
            },
        )
        entities.append(class_entity)
        relations.append(create_relation("HAS_CLASS", parent_entity, class_entity))

        # Process class methods and base classes
        for method_name, method_obj in inspect.getmembers(obj, predicate=inspect.isfunction):
            method_full_path = f"{obj_full_path}.{method_name}"

            params_list = [
                {
                    "name": param_name,
                    "type": str(param.annotation) if param.annotation != inspect.Parameter.empty else "Any",
                    "default": param.default if param.default != inspect.Parameter.empty else None,
                    "kind": str(param.kind)
                }
                for param_name, param in inspect.signature(method_obj).parameters.items()
            ]

            method_entity = create_entity(
                "METHOD",
                method_full_path,
                properties={
                    "comments": inspect.getcomments(method_obj),
                    "doc": inspect.getdoc(method_obj),
                    "signature": str(inspect.signature(method_obj)),
                    "source": inspect.getsource(method_obj),
                    "parameters": params_list,
                },
            )
            entities.append(method_entity)
            relations.append(create_relation("HAS_METHOD", class_entity, method_entity))

        # Base classes
        for base_class in obj.__bases__:
            base_class_entity = create_entity("CLASS", base_class.__name__)
            entities.append(base_class_entity)
            relations.append(create_relation("HAS_BASECLASS", class_entity, base_class_entity))

    elif inspect.isfunction(obj):
        # MODULE-LEVEL FUNCTION entity
        params_list = [
            {
                "name": param_name,
                "type": str(param.annotation) if param.annotation != inspect.Parameter.empty else "Any",
                "default": param.default if param.default != inspect.Parameter.empty else None,
                "kind": str(param.kind)
            }
            for param_name, param in inspect.signature(obj).parameters.items()
        ]

        function_entity = create_entity(
            "METHOD",
            obj_full_path,
            properties={
                "comments": inspect.getcomments(obj),
                "doc": inspect.getdoc(obj),
                "signature": str(inspect.signature(obj)),
                "source": inspect.getsource(obj),
                "parameters": params_list,
            },
        )
        entities.append(function_entity)
        relations.append(create_relation("HAS_METHOD", parent_entity, function_entity))

    elif not name.startswith("__"):  # Ignore dunder attributes
        # Handle constants or other module-level attributes
        if not inspect.isroutine(obj) and not inspect.isclass(obj) and not inspect.ismodule(obj):
            constant_entity = create_entity(
                "CONSTANT",
                obj_full_path,
                properties={
                    "value": repr(obj),
                    "type": type(obj).__name__,
                },
            )
            entities.append(constant_entity)
            relations.append(create_relation("HAS_CONSTANT", parent_entity, constant_entity))

    return entities, relations


# Main function to process a module or object
def process_module_recursive(module_name_or_object):
    """
    Recursively processes a module, class, or object and upserts its structure into Neo4j.
    """
    module_name = ""
    if isinstance(module_name_or_object, str):
        # Handle module-qualified class/object names like "bs4.BeautifulSoup"
        if '.' in module_name_or_object:
            module_name, object_name = module_name_or_object.rsplit('.', 1)
            module = importlib.import_module(module_name)
            module_or_object = getattr(module, object_name)
        else:
            # Assume it's a module name
            module_or_object = importlib.import_module(module_name_or_object)
    else:
        # Assume it's already a module/class/object
        module_or_object = module_name_or_object

    # Traverse the module or object
    entities, relations = traverse_module_recursive(module_or_object)
    upsert_to_neo4j(entities, relations)
    add_module_label(module_name)
    create_module_fts_index(module_name)

In [ ]:
import inspect
import importlib
from typing import Any, Dict, List, Tuple, Union, Set


def safe_get_source(obj) -> str:
    """Retrieve source code or fallback to repr."""
    try:
        return inspect.getsource(obj)
    except (OSError, TypeError):
        return f"# Representation:\n{repr(obj)}"


# def extract_class_constants(cls: Any) -> Dict[str, Any]:
#     """
#     Extract constants and variables directly defined in the class.

#     Args:
#         cls (Any): The class object to extract constants from.

#     Returns:
#         Dict[str, Any]: A dictionary of constants and their string representations.
#     """
#     constants = {}
#     for name, member in inspect.getmembers(cls):
#         # Skip dunder and callable members
#         if name.startswith("_") or inspect.isroutine(member) or inspect.isclass(member):
#             continue
#         constants[name] = repr(member)
#     return constants


def process_member(
    name: str,
    obj: Any,
    parent_entity: Dict[str, Any],
    current_path: str,
    extracted_info: Dict[str, Set[str]],
    parsed_entities: Set[str],
) -> Tuple[List[Dict], List[Dict]]:
    """
    Process individual members of a class or module.

    Args:
        name (str): Name of the member.
        obj (Any): Member object.
        parent_entity (Dict[str, Any]): Parent entity.
        current_path (str): Current traversal path.
        extracted_info (Dict[str, Set[str]]): Extracted info from website parsing.
        parsed_entities (Set[str]): Set of already parsed entities.

    Returns:
        Tuple[List[Dict], List[Dict]]:
            - List of entities
            - List of relations
    """
    entities, relations = [], []
    obj_path = f"{current_path}.{name}"

    # Skip if name starts with '_'
    if name.startswith("_"):
        return [], []

    if inspect.ismodule(obj) or inspect.isclass(obj):
        if obj_path not in extracted_info["module"] and obj_path not in extracted_info["class"]:
            return [], []

        if obj_path in parsed_entities:
            return [], []

        # Recurse into modules or nested classes
        sub_entities, sub_relations = traverse_recursive(
            obj, obj_path, extracted_info, parsed_entities
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

        # Create appropriate hierarchical relation
        relation_type = "HAS_MODULE" if inspect.ismodule(obj) else "HAS_CLASS"
        relations.append(create_relation(relation_type, parent_entity, entities[0]))

    elif inspect.isfunction(obj) or inspect.ismethod(obj):
        # Process function or method
        source_code = safe_get_source(obj)
        # signature = str(inspect.signature(obj))
        method_entity = create_entity(
            "METHOD",
            obj_path,
            {
                "source": source_code,
                # "signature": signature,
            },
        )
        entities.append(method_entity)
        if parent_entity:
            relations.append(create_relation("HAS_METHOD", parent_entity, method_entity))

    return entities, relations


def traverse_recursive(
    obj: Any,
    current_path: str,
    extracted_info: Dict[str, Set[str]],
    parsed_entities: Set[str],
) -> Tuple[List[Dict], List[Dict]]:
    """
    Recursively traverse an object (module or class).

    Args:
        obj (Any): The object to traverse.
        current_path (str): The current traversal path.
        extracted_info (Dict[str, Set[str]]): Extracted info from website parsing.
        parsed_entities (Set[str]): Set of already parsed entities.

    Returns:
        Tuple[List[Dict], List[Dict]]:
            - List of entities
            - List of relations.
    """
    entities, relations = [], []
    obj_name = getattr(obj, "__name__", str(obj))
    entity_type = "MODULE" if inspect.ismodule(obj) else "CLASS"

    # Check if this module/class has already been parsed
    if current_path in parsed_entities:
        return [], []

    # Skip if not in extracted_info
    if (
        entity_type == "MODULE"
        and current_path not in extracted_info["module"]
    ) or (
        entity_type == "CLASS"
        and current_path not in extracted_info["class"]
    ):
        return [], []

    # Create entity for the module or class
    source = safe_get_source(obj)
    properties = {"source": source}
    # constants = {}

    # if entity_type == "CLASS":
    #     # Extract class constants/variables directly defined in the class
    #     constants = extract_class_constants(obj)
    #     properties["constants"] = constants

    entity = create_entity(entity_type, current_path, properties)
    entities.append(entity)
    parsed_entities.add(current_path)  # Mark this entity as parsed

    # Handle inheritance for classes
    if inspect.isclass(obj):
        for base in obj.__bases__:
            if base.__name__ not in ["object", "type"]:
                base_path = f"{base.__module__}.{base.__name__}"
                if base_path in extracted_info["class"] or base_path in extracted_info["module"]:
                    base_entity = create_entity(
                        "CLASS",
                        base_path,
                        {"source": safe_get_source(base)},
                    )
                    entities.append(base_entity)
                    relations.append(create_relation("HAS_BASECLASS", entity, base_entity))

    # Traverse and process members
    for name, member in inspect.getmembers(obj):
        sub_entities, sub_relations = process_member(
            name, member, entity, current_path, extracted_info, parsed_entities
        )
        entities.extend(sub_entities)
        relations.extend(sub_relations)

    return entities, relations


def process_module_recursive(
    module_name_or_object: Union[str, Any], extracted_info: Dict[str, Set[str]]
):
    """
    Process a module or object recursively, traversing only extracted modules/classes.

    Args:
        module_name_or_object (Union[str, Any]): Module name or object.
        extracted_info (Dict[str, Set[str]]): Extracted info from website parsing.
    """
    parsed_entities = set()
    entities, relations = [], []

    # Sort by path length to process shorter paths first (parents before children)
    for item_path in sorted(
        list(extracted_info["module"]) + list(extracted_info["class"]),
        key=len,
    ):
        if item_path in parsed_entities:
            continue  # Skip if already parsed

        try:
            if isinstance(module_name_or_object, str):
                if "." in item_path:
                    module_name, object_name = item_path.rsplit(".", 1)
                    module = importlib.import_module(module_name)
                    module_or_object = getattr(module, object_name)
                else:
                    module_or_object = importlib.import_module(item_path)
            else:
                module_or_object = module_name_or_object

            sub_entities, sub_relations = traverse_recursive(
                module_or_object, item_path, extracted_info, parsed_entities
            )
            entities.extend(sub_entities)
            relations.extend(sub_relations)
        except (ImportError, AttributeError) as e:
            print(f"Error processing {item_path}: {e}")

    print(f"Processed {len(entities)} entities and {len(relations)} relations.")

    return entities, relations


## Try a demo module

In [ ]:
entities, relations = process_module_recursive("requests", extracted_info)

Processed 199 entities and 197 relations.


### Check entities processed

In [ ]:
def print_modules(entities):
  for entity in entities:
    if entity.label == "MODULE":
      print(entity.label, entity.name)

def print_classes(entities):
  for entity in entities:
    if entity.label == "CLASS":
      print(entity.label, entity.name)

def print_methods(entities):
  for entity in entities:
    if entity.label == "METHOD":
      print(entity.label, entity.name)

In [ ]:
print_modules(entities)

MODULE requests
MODULE requests.adapters
MODULE requests.auth
MODULE requests.cookies
MODULE requests.utils


In [ ]:
print_classes(entities)

CLASS requests.ConnectTimeout
CLASS requests.ConnectionError
CLASS requests.HTTPError
CLASS requests.JSONDecodeError
CLASS requests.PreparedRequest
CLASS requests.ReadTimeout
CLASS requests.RequestException
CLASS requests.Response
CLASS requests.Session
CLASS requests.Timeout
CLASS requests.TooManyRedirects
CLASS requests.URLRequired
CLASS requests.adapters.BaseAdapter
CLASS requests.adapters.HTTPAdapter
CLASS requests.adapters.BaseAdapter
CLASS requests.auth.AuthBase
CLASS requests.auth.HTTPBasicAuth
CLASS requests.auth.AuthBase
CLASS requests.auth.HTTPDigestAuth
CLASS requests.auth.AuthBase
CLASS requests.auth.HTTPProxyAuth
CLASS requests.auth.HTTPBasicAuth
CLASS requests.cookies.CookieConflictError
CLASS requests.cookies.RequestsCookieJar
CLASS requests.codes


### Insert the entities and relations into Neo4j

In [ ]:
upsert_to_neo4j(entities, relations)

In [ ]:
add_labels("requests")

# Embed nodes with embeddings

In [ ]:
def prepare_text_for_embedding(node):
    return node.properties["source"]

In [ ]:
# Function to generate embeddings
def generate_embedding(nodes, show_progress=True):
    texts_to_embed = []
    ids_to_embed = [i for i in range(len(nodes))]
    for node in nodes:
      texts_to_embed.append(prepare_text_for_embedding(node))

    new_embeddings = Settings.embed_model.get_text_embedding_batch(
        texts_to_embed, show_progress=show_progress
    )

    for id, text_embedding in zip(ids_to_embed, new_embeddings):
        nodes[id].properties['embedding'] = text_embedding

    return nodes

In [ ]:
def remove_empty_values(input_dict):
    """
    Remove entries with empty values from the dictionary.

    Parameters:
    input_dict (dict): The dictionary from which empty values need to be removed.

    Returns:
    dict: A new dictionary with all empty values removed.
    """
    # Create a new dictionary excluding empty values
    return {key: value for key, value in input_dict.items() if value}

In [ ]:
def get_nodes_without_embeddings(pg_store):
  # Query to fetch nodes that do not have embeddings
  data = pg_store.database_query(
      """
      MATCH (e)
      WHERE e.embedding IS NULL
      RETURN
      e.id AS name,
      labels(e)[0] AS type,
      properties(e) AS properties
      """
  )

  data = data if data else []

  # Create list of EntityNodes
  nodes = []
  for record in data:
      # print(record)
      node = EntityNode(
          name=record["name"],
          label=record["type"],
          properties=remove_empty_values(record["properties"]),
      )
      nodes.append(node)

  return nodes

In [ ]:
# Prepare nodes with generated embeddings
def generate_embedding_for_nodes_without_embeddings(pg_store):
    nodes_without_embeddings = get_nodes_without_embeddings(pg_store)

    if len(nodes_without_embeddings) == 0:
      print("All nodes have embeddings!")
      return

    print("Sample node:", nodes_without_embeddings[0])
    print("Number of nodes without embeddings:", len(nodes_without_embeddings))

    updated_nodes = generate_embedding(nodes_without_embeddings)
    pg_store.upsert_nodes(updated_nodes)

In [ ]:
generate_embedding_for_nodes_without_embeddings(pg_store)

All nodes have embeddings!


# Test the COMStore

## Test querying bs4.BeautifulSoup using a sample prompt

In [ ]:
prompt = "allows easy extraction, navigation and modification of elements"

In [ ]:
prompt_embedding = Settings.embed_model.get_query_embedding(prompt)
print("Length:", len(prompt_embedding))
print("Prompt Embedding:", prompt_embedding)
print()

Length: 1024
Prompt Embedding: [-0.023381052538752556, -0.017727702856063843, -0.009805520065128803, -0.03504539281129837, -0.017339300364255905, 0.03186625987291336, 0.0021248783450573683, -0.0077458214946091175, 0.06094207242131233, 0.034395478665828705, 0.008743264712393284, 0.008832940831780434, 0.010574675165116787, -0.044394902884960175, -0.0069124093279242516, 0.012407802976667881, -0.02181055396795273, -0.0030169771052896976, -0.0474456287920475, 0.024871939793229103, -0.013674530200660229, 0.005256827920675278, -0.0657259076833725, -0.043825000524520874, -0.0185686144977808, 0.015017764642834663, -0.0016810334054753184, 0.0019669365137815475, 0.06715018302202225, 0.05419159680604935, -0.01501681562513113, 0.008195966482162476, -0.008052563294768333, -0.0001021576244966127, -0.013036606833338737, -0.021501747891306877, 0.004506669007241726, -0.03220416605472565, 0.028837064281105995, -0.030001111328601837, 0.03489851951599121, -0.04388444870710373, 0.006571062374860048, -0.0634

In [ ]:
results = pg_store.query_vector_index(
    "class_vector_index",
    query_vector=prompt_embedding
)

In [ ]:
for result in results:
  print(result["name"])

bs4.BeautifulSoup
bs4.SoupStrainer
bs4.Tag
bs4.NavigableString
bs4.UnicodeDammit
bs4.Stylesheet
bs4.formatter.HTMLFormatter
bs4.Doctype
bs4.Script
bs4.Comment


## Test querying bs4.Doctype using a sample prompt

In [ ]:
prompt = "allows inspection and manipulation of DOCTYPE declaration"

In [ ]:
prompt_embedding = Settings.embed_model.get_query_embedding(prompt)
print("Length:", len(prompt_embedding))
print("Prompt Embedding:", prompt_embedding)
print()

Length: 1024
Prompt Embedding: [-0.01496047992259264, -0.06415032595396042, 0.04950472712516785, -0.01700117066502571, -0.024948270991444588, -0.009298915974795818, -0.0005513278883881867, -0.0031504782382398844, -0.030203433707356453, 0.04122619330883026, 0.013718781992793083, 0.00837327353656292, 0.006868468597531319, 0.003317068563774228, -0.020653948187828064, 0.011040214449167252, 0.012148918583989143, -0.017772020772099495, -0.08575187623500824, 0.02304573357105255, -0.01623138040304184, 0.028225770220160484, -0.06853199005126953, 0.004591086879372597, -0.04282040521502495, 0.020735125988721848, 0.00897355005145073, -0.010337792336940765, 0.0726991668343544, 0.04805079847574234, -0.02482391707599163, 0.024730928242206573, -0.024091193452477455, -0.012011650949716568, -0.04226236417889595, -0.06704608350992203, 0.02696847915649414, -0.018441449850797653, 0.04458236321806908, -0.018734417855739594, 0.01603034883737564, -0.01616615429520607, -0.007367504294961691, -0.070540323853492

In [ ]:
results = pg_store.query_vector_index(
    "class_vector_index",
    query_vector=prompt_embedding
)

In [ ]:
for result in results:
  print(result["name"])

bs4.Doctype
bs4.Declaration
bs4.ProcessingInstruction
bs4.UnicodeDammit
bs4.CData
bs4.formatter.HTMLFormatter
bs4.Comment
bs4.Stylesheet
bs4.BeautifulSoup
bs4.SoupStrainer


# Query Planning Workflow

In [ ]:
from pydantic import BaseModel, Field
from llama_index.core.workflow import Event


class QueryPlanItem(Event):
    """A single step in an execution plan for a RAG system."""

    name: str = Field(description="The name of the tool to use.")
    query: str = Field(
        description="A natural language search query for a RAG system."
    )


class QueryPlan(BaseModel):
    """A plan for a RAG system. After running the plan, we should have either enough information to answer the user's original query, or enough information to form a new query plan."""

    items: list[QueryPlanItem] = Field(
        description="A list of the QueryPlanItem objects in the plan."
    )

In [ ]:
class QueryPlanItemResult(Event):
    """The result of a query plan item"""

    query: str
    result: str


class ExecutedPlanEvent(Event):
    """The result of a query plan"""

    result: str

In [ ]:
from llama_index.core.workflow import (
    Workflow,
    StopEvent,
    StartEvent,
    Context,
    step,
)
from llama_index.core.prompts import PromptTemplate
from ratelimit import limits, sleep_and_retry

# Rate limit configuration
RATE_LIMIT = 15
TIME_PERIOD = 60  # seconds


class QueryPlanningWorkflow(Workflow):
    llm = Settings.llm
    planning_prompt = PromptTemplate(
        """
You are an expert programming professional, with access to only an index tool, that will be provided to you in the context/sources (use the name available in that). This tool is used to retrieve information about Python classes, modules, and methods, based on queries. You are tasked with creating a **Query Plan** that systematically breaks down the problem into smaller steps, querying the tool for relevant information to solve the problem effectively.

**Guidelines for Query Plan Construction:**

1. **Understand the Initial Query**: Analyze the provided query to identify what is needed--specific functionality, concepts, or operations.
2. **Use Index Tool Metadata**: Before querying, check the sources available, to confirm if it includes relevant modules, classes, or methods that might assist in answering the query.
   - Avoid querying for modules or functionality that the metadata does not indicate is available.
   - Focus only on retrieving methods, modules, or classes that can help directly in writing Python code.
3. **Structure the Plan**:
   - Each step in the Query Plan should represent a single query aimed at retrieving specific information (e.g., methods for finding tags, replacing them, or handling specific types of data).
   - Ensure that queries are targeted and framed in natural language to explore methods, modules, or functionalities relevant to solving the query. Avoid requesting examples, documentation, or overly general information.
   - Avoid mentioning the specific module/class/function name available in the metadata within your plan items. Frame the queries in natural language to allow discovery of this information.
4. **Iterative Approach**: The Query Plan should aim to retrieve enough information at each step to either resolve the query or refine the next set of queries.

5. **Complete Retrieval of Relevant Nodes**:
   - Do not proceed to generating code or crafting a solution until all relevant methods, classes, or modules (including their signatures, source codes, or other relevant details) have been retrieved using the index tool.
   - Every essential and actionable node should have been retrieved before code generation or crafting the solution.

6. **Substantial Output**: At the end of the Query Plan, the collected results should provide either a direct solution or a clear basis for forming a new, more refined query plan.

---

**Sources Available**: {context}

**Initial Query**: {query}

**Plan**:
        """
    )
    decision_prompt = PromptTemplate(
        "Given the following information, return a final response that satisfies the original query, or return 'PLAN' if you need to continue planning.\n"
        "Original query: {query}\n"
        "Current results: {results}\n"
    )

    @sleep_and_retry
    @limits(calls=RATE_LIMIT, period=TIME_PERIOD)
    async def callLLM(
        self, context_str, query
    ):
      return await self.llm.astructured_predict(
                QueryPlan,
                self.planning_prompt,
                context=context_str,
                query=query,
            )
    @step
    async def planning_step(
        self, ctx: Context, ev: StartEvent | ExecutedPlanEvent
    ) -> QueryPlanItem | StopEvent:
        if isinstance(ev, StartEvent):
            # Initially, we need to plan
            query = ev.get("query")

            tools = ev.get("tools")

            await ctx.set("tools", {t.metadata.name: t for t in tools})
            await ctx.set("original_query", query)

            context_str = "\n".join(
                [
                    f"{i+1}. {tool.metadata.name}: {tool.metadata.description}"
                    for i, tool in enumerate(tools)
                ]
            )
            await ctx.set("context", context_str)

            query_plan = await self.callLLM(context_str, query)

            ctx.write_event_to_stream(
                Event(msg=f"Planning step: {query_plan}")
            )

            num_items = len(query_plan.items)
            await ctx.set("num_items", num_items)
            for item in query_plan.items:
                ctx.send_event(item)
        else:
            # If we've already gone through planning and executing, we need to decide
            # if we should continue planning or if we can stop and return a result.
            query = await ctx.get("original_query")
            current_results_str = ev.result

            decision = await self.llm.apredict(
                self.decision_prompt,
                query=query,
                results=current_results_str,
            )

            # Simple string matching to see if we need to keep planning or if we can stop.
            if "PLAN" in decision:
                context_str = await ctx.get("context")
                query_plan = await self.llm.astructured_predict(
                    QueryPlan,
                    self.planning_prompt,
                    context=context_str,
                    query=query,
                )

                ctx.write_event_to_stream(
                    Event(msg=f"Re-Planning step: {query_plan}")
                )

                num_items = len(query_plan.items)
                await ctx.set("num_items", num_items)
                for item in query_plan.items:
                    ctx.send_event(item)
            else:
                return StopEvent(result=decision)

    @step(num_workers=4)
    async def execute_item(
        self, ctx: Context, ev: QueryPlanItem
    ) -> QueryPlanItemResult:
        tools = await ctx.get("tools")
        tool = tools[ev.name]

        ctx.write_event_to_stream(
            Event(
                msg=f"Querying tool {tool.metadata.name} with query: {ev.query}"
            )
        )

        # Sanitize the query
        sanitized_query = ev.query.replace("/", "\\/")  # Escape special characters
        sanitized_query = sanitized_query.strip()

        result = await tool.acall(sanitized_query)

        ctx.write_event_to_stream(
            Event(msg=f"Tool {tool.metadata.name} returned: {result}")
        )

        return QueryPlanItemResult(query=ev.query, result=str(result))

    @step
    async def aggregate_results(
        self, ctx: Context, ev: QueryPlanItemResult
    ) -> ExecutedPlanEvent:
        # We need to collect the results of the query plan items to aggregate them.
        num_items = await ctx.get("num_items")
        results = ctx.collect_events(ev, [QueryPlanItemResult] * num_items)

        # collect_events returns None if not all events were found
        # return and wait for the remaining events to come in.
        if results is None:
            return

        aggregated_result = "\n------\n".join(
            [
                f"{i+1}. {result.query}: {result.result}"
                for i, result in enumerate(results)
            ]
        )
        return ExecutedPlanEvent(result=aggregated_result)

In [ ]:
index = PropertyGraphIndex.from_existing(
    pg_store
)

In [ ]:
hybrid_context_retriever = HybridContextRetriever(
    pg_store,
    include_text=True,
    include_properties=True,
    similarity_top_k=10,
)

## Use for QueryEngineTool (NOT RECOMMENDED)

In [ ]:
query_engine = index.as_query_engine(
    sub_retrievers = [hybrid_context_retriever],
    verbose=True,
)

In [ ]:
from llama_index.core.tools import QueryEngineTool

query_engine_tools = []
query_engine_tools.append(
    QueryEngineTool.from_defaults(
        query_engine,
        name=f"useful_python_modules",
        description=f"Information about useful python modules, that includes bs4.BeautifulSoup (a powerful web-scraping module)",
    )
)

## Using RetrieverTool

In [ ]:
retriever = index.as_retriever(
    sub_retrievers = [hybrid_context_retriever],
)

In [ ]:
from llama_index.core.tools import RetrieverTool

retriever_tools = [
    RetrieverTool.from_defaults(
      retriever,
      name=f"useful_python_modules",
      description=f"Information about useful python modules. Contains bs4.BeautifulSoup (a powerful web-scraping module)",
    )
]

## Trials

In [ ]:
workflow = QueryPlanningWorkflow(verbose=False, timeout=30)

In [ ]:
prompt = "Write a Python script to scrape and print all the headings (inside <h1> tags) from a webpage."

In [ ]:
# run the workflow
handler = workflow.run(
    query=prompt,
    tools=retriever_tools,
)

# stream the events as they come in
async for event in handler.stream_events():
    if hasattr(event, "msg"):
        print(event.msg)

# get the final result
result = await handler

Planning step: items=[QueryPlanItem(name='useful_python_modules', query='Find a module that can parse HTML and extract tags.'), QueryPlanItem(name='useful_python_modules', query="Within the identified HTML parsing module, find a method to locate all instances of a specific tag, such as 'h1'."), QueryPlanItem(name='useful_python_modules', query='In the identified HTML parsing module, find a method to extract the text content from within the located tags.'), QueryPlanItem(name='useful_python_modules', query='Find a method within the identified module to handle potential errors during HTML parsing, such as malformed HTML.')]
Querying tool useful_python_modules with query: Find a module that can parse HTML and extract tags.
Querying tool useful_python_modules with query: Within the identified HTML parsing module, find a method to locate all instances of a specific tag, such as 'h1'.
Querying tool useful_python_modules with query: In the identified HTML parsing module, find a method to extr

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL () { ... }} {position: line: 1, column: 1, offset: 0} for query: "CALL { CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score WITH collect({node:node, score:score}) AS nodes, max(score) AS max UNWIND nodes AS n RETURN n.node AS node, (n.score / max) AS score UNION CALL db.index.fulltext.queryNodes($keyword_index, $query, {limit: $k}) YIELD node, score WITH collect({node:node, score:score}) AS nodes, max(score) AS max UNWIND nodes AS n RETURN n.node AS node, (n.score / max) AS score } WITH node, max(score) AS score ORDER BY score DESC LIMIT $k RETURN node.id AS name,\n                [l in labels(node) WHERE NOT l IN ['__Entity__', '__Node__'] | l][0] AS ty

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL () { ... }} {position: line: 1, column: 1, offset: 0} for query: "CALL { CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score WITH collect({node:node, score:score}) AS nodes, max(score) AS max UNWIND nodes AS n RETURN n.node AS node, (n.score / max) AS score UNION CALL db.index.fulltext.queryNodes($keyword_index, $query, {limit: $k}) YIELD node, score WITH collect({node:node, score:score}) AS nodes, max(score) AS max UNWIND nodes AS n RETURN n.node AS node, (n.score / max) AS score } WITH node, max(score) AS score ORDER BY score DESC LIMIT $k RETURN node.id AS name,\n                [l in labels(node) WHERE NOT l IN ['__Entity__', '__Node__'] | l][0] AS ty

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL () { ... }} {position: line: 1, column: 1, offset: 0} for query: "CALL { CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score WITH collect({node:node, score:score}) AS nodes, max(score) AS max UNWIND nodes AS n RETURN n.node AS node, (n.score / max) AS score UNION CALL db.index.fulltext.queryNodes($keyword_index, $query, {limit: $k}) YIELD node, score WITH collect({node:node, score:score}) AS nodes, max(score) AS max UNWIND nodes AS n RETURN n.node AS node, (n.score / max) AS score } WITH node, max(score) AS score ORDER BY score DESC LIMIT $k RETURN node.id AS name,\n                [l in labels(node) WHERE NOT l IN ['__Entity__', '__Node__'] | l][0] AS ty

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL () { ... }} {position: line: 1, column: 1, offset: 0} for query: "CALL { CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score WITH collect({node:node, score:score}) AS nodes, max(score) AS max UNWIND nodes AS n RETURN n.node AS node, (n.score / max) AS score UNION CALL db.index.fulltext.queryNodes($keyword_index, $query, {limit: $k}) YIELD node, score WITH collect({node:node, score:score}) AS nodes, max(score) AS max UNWIND nodes AS n RETURN n.node AS node, (n.score / max) AS score } WITH node, max(score) AS score ORDER BY score DESC LIMIT $k RETURN node.id AS name,\n                [l in labels(node) WHERE NOT l IN ['__Entity__', '__Node__'] | l][0] AS ty

In [ ]:
result

'```python\nimport requests\nfrom bs4 import BeautifulSoup\n\ndef scrape_headings(url):\n    """\n    Scrapes and prints all headings (<h1> tags) from a given URL.\n\n    Args:\n        url: The URL of the webpage to scrape.\n    """\n    try:\n        response = requests.get(url)\n        response.raise_for_status()  # Raise an exception for bad status codes\n\n        soup = BeautifulSoup(response.content, \'html.parser\')\n        headings = soup.find_all(\'h1\')\n\n        for heading in headings:\n            print(heading.get_text(strip=True))  # Print the text content, stripping whitespace\n\n    except requests.exceptions.RequestException as e:\n        print(f"An error occurred during the request: {e}")\n    except Exception as e:\n        print(f"An error occurred: {e}")\n\n\nif __name__ == "__main__":\n    url = input("Enter the URL of the webpage: ")\n    scrape_headings(url)\n\n```\n'

# Introspective Agentic Workflow

In [ ]:
from llama_index.agent.introspective import (
    ToolInteractiveReflectionAgentWorker,
)
from llama_index.core.agent import (
    StructuredPlannerAgent,
    ReActAgentWorker
)

## Create Function Tools for pg_store methods

In [ ]:
from llama_index.core.tools import FunctionTool
from llama_index.tools.duckduckgo import DuckDuckGoSearchToolSpec

ddgs_tools = DuckDuckGoSearchToolSpec().to_tool_list()
vector_index_query_tool = FunctionTool.from_defaults(pg_store.query_vector_index)
fulltext_index_query_tool = FunctionTool.from_defaults(pg_store.query_fulltext_index)
similarity_search_without_index_tool = FunctionTool.from_defaults(pg_store.similarity_search_without_index)
database_query_tool = FunctionTool.from_defaults(pg_store.database_query)

all_tools = ddgs_tools + [
    vector_index_query_tool,
    fulltext_index_query_tool,
    similarity_search_without_index_tool,
    database_query_tool,
]

## Create the ReAct Agent

In [ ]:
worker = ReActAgentWorker.from_tools(
    tools = all_tools,
    verbose = True,
)

## Modify the Default Prompts
1) No assumptions

2) Custom module instead of bs4 or requests

3)

In [ ]:
DEFAULT_INITIAL_PLAN_PROMPT = """\
You are an expert at planning and executing tasks related to Python code, especially when you need to interact with a Neo4j knowledge graph containing information about Python modules, classes, and methods.

**Your Goal:**

Given a task, create a comprehensive, step-by-step plan to accomplish it. The plan should prioritize using the database tools to interact with the Neo4j knowledge graph whenever possible, especially for code generation, debugging, and understanding tasks. **Do not make assumptions about what modules or classes might be needed.** Instead, directly query the database to find relevant information. If the database tools are not sufficient, then utilize DuckDuckGo search.

**Tools:**

You have the following tools at your disposal:

{tools_str}

**Database Tool Usage:**

- **Prioritize** using `query_vector_index` or `query_fulltext_index` for initial exploration and finding relevant entities (modules, classes, methods) within the Neo4j database.
- If the index-based search is not sufficient, use `similarity_search_without_index` to perform a broader semantic search across the database.
    - When using `similarity_search_without_index`, use the labels discovered from previous steps (e.g., from `query_vector_index` or `query_fulltext_index`).
    - Be **specific** when providing labels to `similarity_search_without_index`. Use the most specific label available (e.g., "requests.models.Response" instead of just "requests"). Avoid redundant labels.
- Use `database_query` only for highly specific Cypher queries when the other tools are not suitable.
- If the task requires information not available in the database, then use the DuckDuckGo Search tools.

**Hierarchical Approach for Code-Related Queries:**

- When the task involves writing or understanding Python code, adopt a **top-down, hierarchical approach**:
    1. **Modules:** First, identify the relevant modules using the database tools.
    2. **Classes:** Then, within those modules, identify the relevant classes using the database tools.
    3. **Methods:** Finally, focus on the specific methods within those classes that are needed for the task, using the database tools.
- Always consider the inheritance hierarchy (base classes) of identified classes to get a complete understanding by searching for those base classes in the database.

**Planning Guidelines:**

1. **Think Step-by-Step:** Break down the task into logical sub-tasks.
2. **Prioritize Database Tools:** Favor database tools for accessing information about code structure and entities.
3. **No Assumptions:** Do not assume the existence or relevance of specific modules or classes. Always query the database to find them.
4. **Be Comprehensive:** Ensure your plan covers all aspects of the task.
5. **Hierarchical Approach:** For code-related tasks, follow the module -> class -> method hierarchy.
6. **Specific Labels:** Use specific and non-redundant labels when querying with `similarity_search_without_index`.
7. **Fallback to DuckDuckGo:** If the database tools cannot provide the necessary information, use DuckDuckGo search as a fallback.
8. **Clear and Concise:** Make your plan easy to understand and follow. Each sub-task should have a clear objective.
9. **End with a Satisfying Sub-Task:** The final sub-task should directly address and complete the overall task.

**Available Labels in the Database:**

You can use the following labels with the database tools (especially `similarity_search_without_index`):

- `__Entity__`
- `__Node__`
- `MODULE`
- `CLASS`
- `METHOD`
- Names of specific modules available in the database
- Names (full paths) of specific classes available in the database

**Example:**

Let's say the task is: "Generate Python code to make a GET request and extract all headings from the response."

A good plan might look like this:

1. **Find relevant modules:** Use `query_vector_index` or `query_fulltext_index` to find modules related to "making GET requests and extracting headings".
2. **Find relevant classes:** Use `similarity_search_without_index` with labels from step 1, if possible, to find classes within the relevant modules that are related to making GET requests and parsing HTML.
3. **Find relevant methods:** Use `similarity_search_without_index` with labels from the previous steps to identify specific methods for making GET requests and extracting headings.
4. **Get method details:** Use `database_query` (if necessary) to get the details of the relevant methods (parameters, usage, etc.).
5. **Generate code:** Based on the information gathered, generate the Python code.

**Task:**

Overall Task: {task}
"""

DEFAULT_PLAN_REFINE_PROMPT = """\
Think step-by-step. Given an overall task, a set of tools, and completed sub-tasks, update (if needed) the remaining sub-tasks so that the overall task can still be completed.
The plan should end with a sub-task that satisfies the overall task.
If the remaining sub-tasks are sufficient, you can skip this step.

**Prioritize using database tools** for code-related tasks and follow a **hierarchical approach** (module -> class -> method). **Do not make assumptions** about specific modules or classes; query the database directly. Use **specific labels** with `similarity_search_without_index`. Fallback to DuckDuckGo search if the database tools are not sufficient.

The tools available are:
{tools_str}

Overall Task:
{task}

Completed Sub-Tasks + Outputs:
{completed_outputs}

Remaining Sub-Tasks:
{remaining_sub_tasks}
"""

## Create Structured Planning Agent

In [ ]:
agent = StructuredPlannerAgent(
    worker,
    tools=all_tools,
    initial_plan_prompt=DEFAULT_INITIAL_PLAN_PROMPT,
    plan_refine_prompt=DEFAULT_PLAN_REFINE_PROMPT,
    verbose=True
)

In [ ]:
response = await agent.achat(
    "Write a python function to find all headings in an input webpage, provided by the user using a URL."
)

=== Initial plan ===
Find modules for web requests:
Find Python modules related to making HTTP requests. -> A list of module names related to making HTTP requests, such as 'requests'.
deps: []


Find modules for HTML parsing:
Find Python modules related to parsing HTML content. -> A list of module names related to parsing HTML, such as 'bs4'.
deps: []


Find classes for making requests:
Using the modules found in step 1, find classes related to making HTTP requests. -> A list of class names related to making HTTP requests, such as 'requests.Session' or 'requests.api'.
deps: ['Find modules for web requests']


Find classes for HTML parsing:
Using the modules found in step 2, find classes related to parsing HTML content. -> A list of class names related to parsing HTML, such as 'bs4.BeautifulSoup'.
deps: ['Find modules for HTML parsing']


Find methods for making GET requests:
Using the classes found in step 3, find methods for making GET requests. -> A list of method names for making GE

TypeError: object GenerateContentResponse can't be used in 'await' expression